In [ ]:
import os
import re
import json
import time
import math
import random
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple, Any

import numpy as np
import nibabel as nib
from scipy import ndimage as ndi

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

import nibabel as nib
from nibabel.processing import resample_from_to

try:
 from tqdm.auto import tqdm
except Exception:
 def tqdm(iterable=None, *args, **kwargs):
 return iterable if iterable is not None else []


In [ ]:
def mount_drive_in_colab() -> None:
 try:
 from google.colab import drive # type: ignore
 drive.mount('FILL IN WITH DIRECTORY FOR COLAB DRIVE MOUNT', force_remount=False)
 except Exception as exc:
 print(f"Skipping Drive mount: {exc}")

In [ ]:
def set_seed(seed: int = 42) -> None:
 random.seed(seed)
 np.random.seed(seed)
 torch.manual_seed(seed)
 torch.cuda.manual_seed_all(seed)


class AverageMeter:
 def __init__(self) -> None:
 self.reset()

 def reset(self) -> None:
 self.sum = 0.0
 self.count = 0
 self.avg = 0.0

 def update(self, value: float, n: int = 1) -> None:
 self.sum += value * n
 self.count += n
 self.avg = self.sum / max(self.count, 1)


def ensure_dir(path: str) -> None:
 os.makedirs(path, exist_ok=True)


def save_json(obj: Dict, path: str) -> None:
 with open(path, 'w', encoding='utf-8') as f:
 json.dump(obj, f, indent=2)


def load_json(path: str) -> Dict:
 with open(path, 'r', encoding='utf-8') as f:
 return json.load(f)


def is_zone_identifier(path: Path) -> bool:
 return ':Zone.Identifier' in path.name


def case_sort_key(case_id: str) -> Tuple:
 nums = re.findall(r'\d+', case_id)
 return tuple(int(n) for n in nums) if nums else (case_id,)


def count_parameters(model: nn.Module) -> Tuple[int, int]:
 total_params = sum(p.numel() for p in model.parameters())
 trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
 return total_params, trainable_params


def print_parameter_summary(
 model: nn.Module,
 model_name: str = "acf_net",
 baseline_trainable_m: float = 30.43,
 print_modules: bool = True,
) -> None:
 total_params, trainable_params = count_parameters(model)
 total_m = total_params / 1e6
 trainable_m = trainable_params / 1e6
 diff_m = trainable_m - baseline_trainable_m

 print("\n" + "=" * 80)
 print(f"{model_name} parameter summary")
 print("=" * 80)
 print(f"Total parameters: {total_m:.2f}M")
 print(f"Trainable parameters: {trainable_m:.2f}M")
 print(f"Baseline acf_net trainable parameters: {baseline_trainable_m:.2f}M")
 print(f"Difference from baseline: {diff_m:+.2f}M")

 if total_params != trainable_params:
 print(f"Frozen/non-trainable parameters: {(total_params - trainable_params) / 1e6:.2f}M")

 if print_modules:
 print("\nTop-level module parameter counts:")
 for name, module in model.named_children():
 params = sum(p.numel() for p in module.parameters())
 print(f" {name:<28} {params / 1e6:.3f}M")
 print("=" * 80)


def print_gpu_memory(label: str = "") -> None:
 if not torch.cuda.is_available():
 return
 allocated = torch.cuda.memory_allocated() / (1024 ** 3)
 reserved = torch.cuda.memory_reserved() / (1024 ** 3)
 max_allocated = torch.cuda.max_memory_allocated() / (1024 ** 3)
 prefix = f"[GPU memory {label}]" if label else "[GPU memory]"
 print(f"{prefix} allocated={allocated:.2f} GB | reserved={reserved:.2f} GB | max_allocated={max_allocated:.2f} GB")


def print_debug_shapes_once(
 image: torch.Tensor,
 priors: torch.Tensor,
 region: torch.Tensor,
 outputs: Dict[str, torch.Tensor],
 prefix: str = "[DEBUG_SHAPES]",
) -> None:
 print("\n" + "=" * 80)
 print(prefix)
 print("=" * 80)
 print(f"image: {tuple(image.shape)}")
 print(f"priors: {tuple(priors.shape)}")
 print(f"region: {tuple(region.shape)}")
 print(f"logits_expert_1: {tuple(outputs['logits_expert_1'].shape)}")
 print(f"logits_expert_2: {tuple(outputs['logits_expert_2'].shape)}")
 print(f"logits_expert_3: {tuple(outputs['logits_expert_3'].shape)}")
 print(f"region_weights: {tuple(outputs['region_weights'].shape)}")
 print(f"router_scores: {tuple(outputs['router_scores'].shape)}")
 print(f"image-level weights: {tuple(outputs['weights'].shape)}")
 if "voxel_weights" in outputs:
 print(f"voxel_weights: {tuple(outputs['voxel_weights'].shape)}")
 print(f"logits_fused: {tuple(outputs['logits_fused'].shape)}")
 print("=" * 80 + "\n")


In [ ]:
@dataclass
class TrainConfig:
 smoke_test: bool = False
 smoke_test_epochs: int = 2
 precompute_all_priors_before_training: bool = False

 # Dataset roots. Fill these in before running the notebook.
 brats_root: str = 'FILL IN WITH DIRECTORY FOR BRATS GLI TRAINING DATA'
 fastsurfer_root: str = 'FILL IN WITH DIRECTORY FOR FASTSURFER TRAINING OUTPUTS'

 # acf_net output/cache/checkpoint paths.
 # Keep these separate from other experiment files.
 work_dir: str = 'FILL IN WITH DIRECTORY FOR ACF NET BRATS RUNS'
 cache_dir: str = 'FILL IN WITH DIRECTORY FOR VALIDATION PRIOR CACHE'
 checkpoint_dir: str = 'FILL IN WITH DIRECTORY FOR MODEL CHECKPOINTS'
 log_dir: str = 'FILL IN WITH DIRECTORY FOR TRAINING LOGS'
 manifest_path: str = 'FILL IN WITH DIRECTORY FOR RUN MANIFEST/manifest.json'
 split_path: str = 'FILL IN WITH DIRECTORY FOR TRAIN VALIDATION TEST SPLITS/splits.json'

 # Split generation
 split_seed: int = 42
 train_ratio: float = 0.6
 val_ratio: float = 0.2
 test_ratio: float = 0.2
 regenerate_manifest: bool = False
 regenerate_splits: bool = False

 # Data
 num_classes: int = 4
 batch_size: int = 2
 num_workers: int = 2
 pin_memory: bool = True
 persistent_workers: bool = True
 prefetch_factor: int = 2
 patch_size: Tuple[int, int, int] = (96, 96, 96)
 use_patch_sampling: bool = False
 cache_priors: bool = True
 debug_resampling_shapes: bool = False

 # acf_net router settings
 num_regions: int = 6
 region_vocab: int = 16
 router_hidden: int = 32
 router_region_dim: int = 8
 router_global_dim: int = 16
 router_calibration_lambda: float = 1.0

 # Training
 seed: int = 42
 lr: float = 2e-4
 weight_decay: float = 1e-5
 max_epochs: int = 300
 patience: int = 25
 monitor_mode: str = 'max'
 monitor_metric: str = 'val_mean_dice'
 min_delta: float = 1e-4
 amp: bool = True
 grad_clip_norm: float = 12.0
 device: str = 'cuda' if torch.cuda.is_available() else 'cpu'

 # Checkpointing / resume
 save_every: int = 1
 keep_last_k: int = 3
 resume_path: Optional[str] = None

 # Explicit acf_net engineering flags
 AUTO_RESUME: bool = True
 DEBUG_SHAPES: bool = True
 SAVE_EVERY_EPOCH: bool = True
 SAVE_BEST_ONLY: bool = False


In [ ]:
BRATS_MODALITY_SUFFIXES = {
 't1': 't1n',
 't1ce': 't1c',
 't2': 't2w',
 'flair': 't2f',
}

FASTSURFER_FILES = {
 'dkt': 'aparc.DKTatlas+aseg.deep.mgz',
 'aseg': 'aseg.auto_noCCseg.mgz',
 'mask': 'mask.mgz',
}

In [ ]:
def build_case_record(case_id: str, brats_root: Path, fastsurfer_root: Path) -> Optional[Dict]:
 brats_case_dir = brats_root / case_id
 fs_case_mri_dir = fastsurfer_root / case_id / 'mri'

 if not brats_case_dir.is_dir() or not fs_case_mri_dir.is_dir():
 return None

 files = {
 't1': brats_case_dir / f'{case_id}-{BRATS_MODALITY_SUFFIXES["t1"]}.nii.gz',
 't1ce': brats_case_dir / f'{case_id}-{BRATS_MODALITY_SUFFIXES["t1ce"]}.nii.gz',
 't2': brats_case_dir / f'{case_id}-{BRATS_MODALITY_SUFFIXES["t2"]}.nii.gz',
 'flair': brats_case_dir / f'{case_id}-{BRATS_MODALITY_SUFFIXES["flair"]}.nii.gz',
 'seg': brats_case_dir / f'{case_id}-seg.nii.gz',
 'dkt': fs_case_mri_dir / FASTSURFER_FILES['dkt'],
 'aseg': fs_case_mri_dir / FASTSURFER_FILES['aseg'],
 'mask': fs_case_mri_dir / FASTSURFER_FILES['mask'],
 }

 missing = [k for k, p in files.items() if (not p.exists()) or is_zone_identifier(p)]
 if missing:
 return None

 return {
 'id': case_id,
 'brats_dir': str(brats_case_dir),
 'fastsurfer_mri_dir': str(fs_case_mri_dir),
 'mri': {
 't1': str(files['t1']),
 't1ce': str(files['t1ce']),
 't2': str(files['t2']),
 'flair': str(files['flair']),
 },
 'seg': str(files['seg']),
 'fastsurfer': {
 'dkt': str(files['dkt']),
 'aseg': str(files['aseg']),
 'mask': str(files['mask']),
 },
 }


def scan_and_match_cases(cfg: TrainConfig) -> List[Dict]:
 brats_root = Path(cfg.brats_root)
 fastsurfer_root = Path(cfg.fastsurfer_root)

 brats_case_ids = {
 p.name for p in brats_root.iterdir()
 if p.is_dir() and p.name.startswith('BraTS-GLI-') and not is_zone_identifier(p)
 }
 fs_case_ids = {
 p.name for p in fastsurfer_root.iterdir()
 if p.is_dir() and p.name.startswith('BraTS-GLI-') and not is_zone_identifier(p)
 }

 shared_case_ids = sorted(brats_case_ids.intersection(fs_case_ids), key=case_sort_key)
 manifest: List[Dict] = []

 for case_id in shared_case_ids:
 rec = build_case_record(case_id, brats_root, fastsurfer_root)
 if rec is not None:
 manifest.append(rec)

 return manifest


def load_or_create_manifest(cfg: TrainConfig) -> List[Dict]:
 ensure_dir(os.path.dirname(cfg.manifest_path))
 if os.path.exists(cfg.manifest_path) and not cfg.regenerate_manifest:
 data = load_json(cfg.manifest_path)
 return data['cases']

 cases = scan_and_match_cases(cfg)
 save_json({'cases': cases}, cfg.manifest_path)
 return cases

In [ ]:
def generate_splits_from_manifest(cfg: TrainConfig, cases: List[Dict]) -> Dict[str, List[Dict]]:
 ids = list(range(len(cases)))
 rng = random.Random(cfg.split_seed)
 rng.shuffle(ids)

 n = len(ids)
 n_train = int(round(n * cfg.train_ratio))
 n_val = int(round(n * cfg.val_ratio))
 n_test = n - n_train - n_val

 if n_test < 1 and n >= 3:
 n_test = 1
 if n_train > n_val:
 n_train -= 1
 else:
 n_val -= 1

 train_idx = ids[:n_train]
 val_idx = ids[n_train:n_train + n_val]
 test_idx = ids[n_train + n_val:]

 splits = {
 'train': [cases[i] for i in train_idx],
 'val': [cases[i] for i in val_idx],
 'test': [cases[i] for i in test_idx],
 }
 return splits


def load_or_create_splits(cfg: TrainConfig, cases: List[Dict]) -> Dict[str, List[Dict]]:
 ensure_dir(os.path.dirname(cfg.split_path))
 if os.path.exists(cfg.split_path) and not cfg.regenerate_splits:
 return load_json(cfg.split_path)

 splits = generate_splits_from_manifest(cfg, cases)
 save_json(splits, cfg.split_path)
 return splits

In [ ]:
def load_nib(path: str) -> nib.spatialimages.SpatialImage:
 return nib.load(path)


def load_volume(path: str, dtype=np.float32) -> np.ndarray:
 img = load_nib(path)
 arr = np.asanyarray(img.dataobj)
 return arr.astype(dtype, copy=False)


def load_volume_with_meta(path: str, dtype=np.float32):
 img = load_nib(path)
 arr = np.asanyarray(img.dataobj).astype(dtype, copy=False)
 return arr, img.affine, img.header, img


def zscore_nonzero(volume: np.ndarray) -> np.ndarray:
 volume = volume.astype(np.float32)
 mask = volume != 0
 if mask.sum() == 0:
 return volume
 vals = volume[mask]
 mean = float(vals.mean())
 std = float(vals.std())
 std = max(std, 1e-6)
 out = volume.copy()
 out[mask] = (out[mask] - mean) / std
 return out


def remap_brats_labels(seg: np.ndarray) -> np.ndarray:
 seg = seg.astype(np.int64)
 # BraTS 2023 labels commonly use 0,1,2,3 already in GLI format.
 # This keeps them unchanged. If data contains 4, map 4->3.
 seg = seg.copy()
 seg[seg == 4] = 3
 return seg

In [ ]:
def _make_image_from_array(array: np.ndarray, affine: np.ndarray, header=None, is_label: bool = False):
 """
 Wrap a numpy array in a nib image for resampling.
 """
 if is_label:
 array = array.astype(np.int16, copy=False)
 else:
 array = array.astype(np.float32, copy=False)
 return nib.Nifti1Image(array, affine, header=header)


def resample_array_to_reference(
 src_array: np.ndarray,
 src_affine: np.ndarray,
 ref_img: nib.spatialimages.SpatialImage,
 order: int,
 src_header=None,
) -> np.ndarray:
 """
 Resample a single 3D array into the reference image space.
 order=1 for continuous maps
 order=0 for discrete label maps
 """
 src_img = _make_image_from_array(src_array, src_affine, header=src_header, is_label=(order == 0))
 out_img = resample_from_to(src_img, ref_img, order=order)
 out = np.asanyarray(out_img.dataobj)

 if order == 0:
 return np.rint(out).astype(np.int32)
 return out.astype(np.float32, copy=False)


def resample_priors_and_region_to_brats_space(
 priors: np.ndarray,
 region: np.ndarray,
 src_affine: np.ndarray,
 ref_img: nib.spatialimages.SpatialImage,
 src_header=None,
):
 """
 priors: [C, Z, Y, X] continuous channels
 region: [Z, Y, X] discrete coarse region map
 """
 resampled_priors = []
 for c in range(priors.shape[0]):
 resampled_c = resample_array_to_reference(
 src_array=priors[c],
 src_affine=src_affine,
 ref_img=ref_img,
 order=1,
 src_header=src_header,
 )
 resampled_priors.append(resampled_c)

 resampled_priors = np.stack(resampled_priors, axis=0).astype(np.float32, copy=False)

 resampled_region = resample_array_to_reference(
 src_array=region,
 src_affine=src_affine,
 ref_img=ref_img,
 order=0,
 src_header=src_header,
 )

 return resampled_priors, resampled_region

In [ ]:
def derive_anatomy_priors_in_fastsurfer_space(case: Dict):
 """
 Build anatomy priors and coarse region map in native FastSurfer space.

 Returns:
 priors_fs: np.ndarray [C, Z, Y, X]
 region_fs: np.ndarray [Z, Y, X]
 fs_affine: np.ndarray
 fs_header: nib header
 """
 dkt_img = load_nib(case['fastsurfer']['dkt'])
 aseg_img = load_nib(case['fastsurfer']['aseg'])
 mask_img = load_nib(case['fastsurfer']['mask'])

 dkt = np.asanyarray(dkt_img.dataobj).astype(np.int32, copy=False)
 aseg = np.asanyarray(aseg_img.dataobj).astype(np.int32, copy=False)
 mask = np.asanyarray(mask_img.dataobj).astype(np.float32, copy=False)

 wm = np.isin(aseg, [2, 41]).astype(np.float32)
 ventricle = np.isin(aseg, [4, 5, 14, 15, 43, 44]).astype(np.float32)
 deep_gray = np.isin(aseg, [10, 11, 12, 13, 17, 18, 26, 49, 50, 51, 52, 53, 54, 58]).astype(np.float32)
 cortex = ((dkt >= 1000) & (dkt < 4000)).astype(np.float32)
 csf = np.logical_or(ventricle > 0, np.isin(aseg, [24]).astype(bool)).astype(np.float32)

 gm = np.logical_or(cortex > 0, deep_gray > 0).astype(np.float32)

 # Restrict tissue priors to mask if desired.
 brain_mask = (mask > 0).astype(np.float32)
 wm *= brain_mask
 gm *= brain_mask
 csf *= brain_mask
 ventricle *= brain_mask
 deep_gray *= brain_mask
 cortex *= brain_mask

 # Distance maps in FastSurfer space.
 wm_dist = ndi.distance_transform_edt(1.0 - wm).astype(np.float32)
 vent_dist = ndi.distance_transform_edt(1.0 - ventricle).astype(np.float32)

 # Boundary map from anatomical segmentation.
 anat_labels = dkt.copy()
 anat_boundary = np.zeros_like(anat_labels, dtype=bool)
 anat_boundary[:-1, :, :] |= (anat_labels[:-1, :, :] != anat_labels[1:, :, :])
 anat_boundary[:, :-1, :] |= (anat_labels[:, :-1, :] != anat_labels[:, 1:, :])
 anat_boundary[:, :, :-1] |= (anat_labels[:, :, :-1] != anat_labels[:, :, 1:])
 anat_boundary = anat_boundary.astype(np.float32) * brain_mask

 # Coarse region map
 # 0 background / outside mask
 # 1 cortex
 # 2 white matter
 # 3 ventricles / CSF
 # 4 deep gray
 # 5 other brain
 region = np.zeros_like(dkt, dtype=np.int32)
 region[brain_mask > 0] = 5
 region[wm > 0] = 2
 region[gm > 0] = 1
 region[deep_gray > 0] = 4
 region[np.logical_or(ventricle > 0, csf > 0)] = 3
 region[brain_mask == 0] = 0

 priors_fs = np.stack(
 [
 wm,
 gm,
 csf,
 ventricle,
 deep_gray,
 cortex,
 wm_dist,
 vent_dist,
 anat_boundary,
 ],
 axis=0,
 ).astype(np.float32, copy=False)

 return priors_fs, region, dkt_img.affine, dkt_img.header

In [ ]:
def normalize_distance_channel(x: np.ndarray) -> np.ndarray:
 x = x.astype(np.float32, copy=False)
 x_min = float(x.min())
 x_max = float(x.max())
 if x_max - x_min < 1e-8:
 return np.zeros_like(x, dtype=np.float32)
 return (x - x_min) / (x_max - x_min)

In [ ]:
VENTRICLE_LABELS = {4, 5, 14, 15, 24, 31, 43, 44, 63}
DEEP_GRAY_LABELS = {10, 11, 12, 13, 17, 18, 26, 49, 50, 51, 52, 53, 54, 58}
WM_LABELS = {2, 41, 77, 78, 79}
GM_LABELS = {
 3, 8, 9, 10, 11, 12, 13, 16, 17, 18, 26,
 42, 47, 48, 49, 50, 51, 52, 53, 54, 58,
}
CSF_LABELS = {4, 5, 14, 15, 24, 31, 43, 44, 63}


# DKT cortical labels are broad ranges. This lightweight rule is sufficient
# for building soft anatomy context maps in this implementation.
def is_cortex_label(x: np.ndarray) -> np.ndarray:
 return ((x >= 1000) & (x < 3000)).astype(np.float32)


def compute_boundary(binary_mask: np.ndarray) -> np.ndarray:
 binary_mask = binary_mask.astype(bool)
 if not binary_mask.any():
 return np.zeros_like(binary_mask, dtype=np.float32)
 eroded = ndi.binary_erosion(binary_mask, iterations=1, border_value=0)
 boundary = binary_mask ^ eroded
 return boundary.astype(np.float32)


def normalized_distance(mask: np.ndarray) -> np.ndarray:
 mask = mask.astype(bool)
 if not mask.any():
 return np.zeros_like(mask, dtype=np.float32)
 dist = ndi.distance_transform_edt(~mask).astype(np.float32)
 maxv = float(dist.max())
 if maxv > 0:
 dist /= maxv
 return dist


def build_coarse_region_map(dkt: np.ndarray, aseg: np.ndarray, ventricle: np.ndarray, cortex: np.ndarray) -> np.ndarray:
 region = np.zeros_like(aseg, dtype=np.int64)

 deep_gray = np.isin(aseg, list(DEEP_GRAY_LABELS))
 wm = np.isin(aseg, list(WM_LABELS))

 left_mask = ((aseg >= 1) & (aseg < 40)) | ((dkt >= 1000) & (dkt < 2000))
 right_mask = ((aseg >= 40) & (aseg < 80)) | ((dkt >= 2000) & (dkt < 3000))
 periventricular = ndi.binary_dilation(ventricle.astype(bool), iterations=3) & (~ventricle.astype(bool))

 region[left_mask & wm] = 1
 region[right_mask & wm] = 2
 region[cortex.astype(bool)] = 3
 region[periventricular] = 4
 region[deep_gray] = 5
 return region


def derive_anatomy_priors_from_fastsurfer(case: Dict, verbose: bool = False):
 """
 Build priors in FastSurfer space, then resample them into BRATS T1 space.
 """
 priors_fs, region_fs, fs_affine, fs_header = derive_anatomy_priors_in_fastsurfer_space(case)

 # BRATS T1 is the reference space.
 ref_img = load_nib(case['mri']['t1'])

 priors_brats, region_brats = resample_priors_and_region_to_brats_space(
 priors=priors_fs,
 region=region_fs,
 src_affine=fs_affine,
 ref_img=ref_img,
 src_header=fs_header,
 )

 # Normalize distance-map channels after resampling into BRATS space.
 # Channel 6 = wm_dist
 # Channel 7 = vent_dist
 priors_brats[6] = normalize_distance_channel(priors_brats[6])
 priors_brats[7] = normalize_distance_channel(priors_brats[7])

 if verbose:
 ref_shape = np.asanyarray(ref_img.dataobj).shape
 print(f"[Resampling check] MRI shape: {ref_shape}")
 print(f"[Resampling check] resampled priors shape: {priors_brats.shape}")
 print(f"[Resampling check] resampled region shape: {region_brats.shape}")

 return priors_brats, region_brats


def get_cached_prior_paths(cfg: TrainConfig, case_id: str) -> Tuple[str, str]:
 case_dir = os.path.join(cfg.cache_dir, case_id)
 ensure_dir(case_dir)
 return os.path.join(case_dir, 'priors.npy'), os.path.join(case_dir, 'region.npy')


def load_or_build_priors(cfg: TrainConfig, case: Dict):
 case_cache_dir = os.path.join(cfg.cache_dir, case['id'])
 priors_cache = os.path.join(case_cache_dir, 'priors.npy')
 region_cache = os.path.join(case_cache_dir, 'region.npy')

 if cfg.cache_priors and os.path.exists(priors_cache) and os.path.exists(region_cache):
 try:
 priors = np.load(priors_cache).astype(np.float32, copy=False)
 region = np.load(region_cache).astype(np.int32, copy=False)
 return priors, region
 except Exception as e:
 print(
 f"[Warning] Cache load failed for case {case['id']}. "
 f"Recomputing priors and rewriting cache. Error: {repr(e)}"
 )

 priors, region = derive_anatomy_priors_from_fastsurfer(case, verbose=False)

 if cfg.cache_priors:
 os.makedirs(case_cache_dir, exist_ok=True)
 np.save(priors_cache, priors)
 np.save(region_cache, region)

 return priors, region

def precompute_all_priors(cfg: TrainConfig, cases: List[Dict]) -> None:
 ensure_dir(cfg.cache_dir)
 for idx, case in enumerate(cases):
 load_or_build_priors(cfg, case)
 if (idx + 1) % 25 == 0 or (idx + 1) == len(cases):
 print(f'Cached priors for {idx + 1}/{len(cases)} cases')

In [ ]:
class BratsFastSurferDataset(Dataset):
 def __init__(
 self,
 cfg: TrainConfig,
 samples: Sequence[Dict],
 training: bool = True,
 ) -> None:
 self.cfg = cfg
 self.samples = list(samples)
 self.training = training

 def __len__(self) -> int:
 return len(self.samples)

 def _pad_if_needed(self, arr: np.ndarray, target: Tuple[int, int, int], value: float = 0) -> np.ndarray:
 dz = max(target[0] - arr.shape[-3], 0)
 dy = max(target[1] - arr.shape[-2], 0)
 dx = max(target[2] - arr.shape[-1], 0)
 if dz == 0 and dy == 0 and dx == 0:
 return arr
 pad_width = [(0, 0)] * arr.ndim
 pad_width[-3] = (dz // 2, dz - dz // 2)
 pad_width[-2] = (dy // 2, dy - dy // 2)
 pad_width[-1] = (dx // 2, dx - dx // 2)
 return np.pad(arr, pad_width, mode='constant', constant_values=value)

 def _center_crop(self, arr: np.ndarray, target: Tuple[int, int, int]) -> np.ndarray:
 z, y, x = arr.shape[-3:]
 tz, ty, tx = target
 sz = max((z - tz) // 2, 0)
 sy = max((y - ty) // 2, 0)
 sx = max((x - tx) // 2, 0)
 return arr[..., sz:sz + tz, sy:sy + ty, sx:sx + tx]

 def _random_crop(self, arr: np.ndarray, start: Tuple[int, int, int], target: Tuple[int, int, int]) -> np.ndarray:
 sz, sy, sx = start
 tz, ty, tx = target
 return arr[..., sz:sz + tz, sy:sy + ty, sx:sx + tx]

 def _sample_start(self, shape: Tuple[int, int, int], target: Tuple[int, int, int]) -> Tuple[int, int, int]:
 starts = []
 for dim, tgt in zip(shape, target):
 if dim <= tgt:
 starts.append(0)
 else:
 starts.append(random.randint(0, dim - tgt))
 return tuple(starts)

 def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
 sample = self.samples[idx]

 t1 = zscore_nonzero(load_volume(sample['mri']['t1'], dtype=np.float32))
 t1ce = zscore_nonzero(load_volume(sample['mri']['t1ce'], dtype=np.float32))
 t2 = zscore_nonzero(load_volume(sample['mri']['t2'], dtype=np.float32))
 flair = zscore_nonzero(load_volume(sample['mri']['flair'], dtype=np.float32))
 seg = remap_brats_labels(load_volume(sample['seg'], dtype=np.int32)).astype(np.int64)

 x = np.stack([t1, t1ce, t2, flair], axis=0).astype(np.float32, copy=False)

 # Priors + region should already be derived and resampled into BRATS space
 priors, region = load_or_build_priors(self.cfg, sample)

 # Optional one-time debug print for shape alignment
 if idx == 0 and getattr(self.cfg, 'debug_resampling_shapes', False):
 print(f"[Dataset check] MRI shape: {x.shape[1:]}")
 print(f"[Dataset check] seg shape: {seg.shape}")
 print(f"[Dataset check] resampled priors shape: {priors.shape}")
 print(f"[Dataset check] resampled region shape: {region.shape}")

 x = self._pad_if_needed(x, self.cfg.patch_size, value=0)
 priors = self._pad_if_needed(priors, self.cfg.patch_size, value=0)
 seg = self._pad_if_needed(seg, self.cfg.patch_size, value=0)
 region = self._pad_if_needed(region, self.cfg.patch_size, value=0)

 if self.cfg.use_patch_sampling and self.training:
 start = self._sample_start(seg.shape[-3:], self.cfg.patch_size)
 x = self._random_crop(x, start, self.cfg.patch_size)
 priors = self._random_crop(priors, start, self.cfg.patch_size)
 seg = self._random_crop(seg, start, self.cfg.patch_size)
 region = self._random_crop(region, start, self.cfg.patch_size)
 else:
 x = self._center_crop(x, self.cfg.patch_size)
 priors = self._center_crop(priors, self.cfg.patch_size)
 seg = self._center_crop(seg, self.cfg.patch_size)
 region = self._center_crop(region, self.cfg.patch_size)

 return {
 'id': sample['id'],
 'image': torch.from_numpy(x).float(),
 'priors': torch.from_numpy(priors).float(),
 'region': torch.from_numpy(region).long(),
 'seg': torch.from_numpy(seg).long(),
 }

In [ ]:
class ConvNormAct(nn.Module):
 def __init__(self, in_ch: int, out_ch: int, k: int = 3, s: int = 1, p: int = 1):
 super().__init__()
 self.block = nn.Sequential(
 nn.Conv3d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, bias=False),
 nn.InstanceNorm3d(out_ch),
 nn.LeakyReLU(0.01, inplace=True),
 )

 def forward(self, x: torch.Tensor) -> torch.Tensor:
 return self.block(x)


class ResidualBlock3D(nn.Module):
 def __init__(self, in_ch: int, out_ch: int, stride: int = 1):
 super().__init__()
 self.conv1 = ConvNormAct(in_ch, out_ch, k=3, s=stride, p=1)
 self.conv2 = nn.Sequential(
 nn.Conv3d(out_ch, out_ch, kernel_size=3, stride=1, padding=1, bias=False),
 nn.InstanceNorm3d(out_ch),
 )
 if in_ch != out_ch or stride != 1:
 self.skip = nn.Sequential(
 nn.Conv3d(in_ch, out_ch, kernel_size=1, stride=stride, bias=False),
 nn.InstanceNorm3d(out_ch),
 )
 else:
 self.skip = nn.Identity()
 self.act = nn.LeakyReLU(0.01, inplace=True)

 def forward(self, x: torch.Tensor) -> torch.Tensor:
 identity = self.skip(x)
 out = self.conv1(x)
 out = self.conv2(out)
 out = self.act(out + identity)
 return out


class MRIEncoder3D(nn.Module):
 def __init__(self, in_channels: int = 4, base: int = 32):
 super().__init__()
 self.e0 = nn.Sequential(ConvNormAct(in_channels, base), ResidualBlock3D(base, base))
 self.e1 = ResidualBlock3D(base, base * 2, stride=2)
 self.e2 = ResidualBlock3D(base * 2, base * 4, stride=2)
 self.e3 = ResidualBlock3D(base * 4, base * 8, stride=2)
 self.e4 = ResidualBlock3D(base * 8, 320, stride=2)

 def forward(self, x: torch.Tensor) -> List[torch.Tensor]:
 f0 = self.e0(x)
 f1 = self.e1(f0)
 f2 = self.e2(f1)
 f3 = self.e3(f2)
 f4 = self.e4(f3)
 return [f0, f1, f2, f3, f4]


class AnatomyEncoder3D(nn.Module):
 def __init__(self, in_channels: int = 9, base: int = 16):
 super().__init__()
 self.a0 = nn.Sequential(ConvNormAct(in_channels, base), ResidualBlock3D(base, base))
 self.a1 = ResidualBlock3D(base, base * 2, stride=2)
 self.a2 = ResidualBlock3D(base * 2, base * 4, stride=2)
 self.a3 = ResidualBlock3D(base * 4, base * 8, stride=2)
 self.a4 = ResidualBlock3D(base * 8, 160, stride=2)

 def forward(self, x: torch.Tensor) -> List[torch.Tensor]:
 g0 = self.a0(x)
 g1 = self.a1(g0)
 g2 = self.a2(g1)
 g3 = self.a3(g2)
 g4 = self.a4(g3)
 return [g0, g1, g2, g3, g4]


class AnatomyModulationBlock(nn.Module):
 def __init__(self, feat_ch: int, anat_ch: int):
 super().__init__()
 self.anat_proj = nn.Conv3d(anat_ch, feat_ch, kernel_size=1)
 self.spatial = nn.Sequential(
 nn.Conv3d(feat_ch * 2, feat_ch, kernel_size=3, padding=1),
 nn.InstanceNorm3d(feat_ch),
 nn.LeakyReLU(0.01, inplace=True),
 nn.Conv3d(feat_ch, feat_ch, kernel_size=1),
 nn.Sigmoid(),
 )
 hidden = max(feat_ch // 2, 16)
 self.film = nn.Sequential(
 nn.Linear(anat_ch, hidden),
 nn.ReLU(inplace=True),
 nn.Linear(hidden, feat_ch * 2),
 )
 self.refine = ResidualBlock3D(feat_ch, feat_ch)

 def forward(self, x: torch.Tensor, anat: torch.Tensor) -> torch.Tensor:
 anat_p = self.anat_proj(anat)
 attn = self.spatial(torch.cat([x, anat_p], dim=1))
 out = x * (1.0 + attn)
 z = F.adaptive_avg_pool3d(anat, 1).flatten(1)
 gamma_beta = self.film(z)
 gamma, beta = torch.chunk(gamma_beta, 2, dim=1)
 gamma = gamma.unsqueeze(-1).unsqueeze(-1).unsqueeze(-1)
 beta = beta.unsqueeze(-1).unsqueeze(-1).unsqueeze(-1)
 out = gamma * out + beta
 out = self.refine(out)
 return out


class DecoderBlock3D(nn.Module):
 def __init__(self, in_ch: int, skip_ch: int, out_ch: int, anat_ch: int):
 super().__init__()
 self.up = nn.ConvTranspose3d(in_ch, out_ch, kernel_size=2, stride=2)
 self.merge = ResidualBlock3D(out_ch + skip_ch, out_ch)
 self.mod = AnatomyModulationBlock(out_ch, anat_ch)

 def forward(self, x: torch.Tensor, skip: torch.Tensor, anat: torch.Tensor) -> torch.Tensor:
 x = self.up(x)
 if x.shape[-3:] != skip.shape[-3:]:
 x = F.interpolate(x, size=skip.shape[-3:], mode='trilinear', align_corners=False)
 x = torch.cat([x, skip], dim=1)
 x = self.merge(x)
 x = self.mod(x, anat)
 return x


class SharedTrunkHead(nn.Module):
 def __init__(self, in_ch: int = 32, out_ch: int = 64):
 super().__init__()
 self.block = nn.Sequential(ResidualBlock3D(in_ch, in_ch), ConvNormAct(in_ch, out_ch, k=3, s=1, p=1))

 def forward(self, x: torch.Tensor) -> torch.Tensor:
 return self.block(x)


class ContextExpertHead(nn.Module):
 def __init__(self, trunk_ch: int = 64, context_ch: int = 128, num_classes: int = 4):
 super().__init__()
 self.compress = ConvNormAct(trunk_ch + context_ch, 64)
 self.refine = ResidualBlock3D(64, 64)
 self.out = nn.Conv3d(64, num_classes, kernel_size=1)

 def forward(self, trunk: torch.Tensor, context: torch.Tensor) -> torch.Tensor:
 if context.shape[-3:] != trunk.shape[-3:]:
 context = F.interpolate(context, size=trunk.shape[-3:], mode='trilinear', align_corners=False)
 x = torch.cat([trunk, context], dim=1)
 x = self.compress(x)
 x = self.refine(x)
 return self.out(x)


class BoundaryExpertHead(nn.Module):
 def __init__(self, trunk_ch: int = 64, boundary_ch: int = 16, edge_ch: int = 16, num_classes: int = 4):
 super().__init__()
 self.edge_proj = ConvNormAct(32, edge_ch)
 self.boundary_proj = ConvNormAct(boundary_ch, boundary_ch)
 self.refine1 = ResidualBlock3D(trunk_ch + boundary_ch + edge_ch, 64)
 self.refine2 = ResidualBlock3D(64, 64)
 self.out = nn.Conv3d(64, num_classes, kernel_size=1)

 def forward(self, trunk: torch.Tensor, early_feat: torch.Tensor, boundary_feat: torch.Tensor) -> torch.Tensor:
 edge = self.edge_proj(early_feat)
 if edge.shape[-3:] != trunk.shape[-3:]:
 edge = F.interpolate(edge, size=trunk.shape[-3:], mode='trilinear', align_corners=False)
 boundary = self.boundary_proj(boundary_feat)
 if boundary.shape[-3:] != trunk.shape[-3:]:
 boundary = F.interpolate(boundary, size=trunk.shape[-3:], mode='trilinear', align_corners=False)
 x = torch.cat([trunk, edge, boundary], dim=1)
 x = self.refine1(x)
 x = self.refine2(x)
 return self.out(x)


class SEBlock3D(nn.Module):
 def __init__(self, channels: int, reduction: int = 8):
 super().__init__()
 hidden = max(channels // reduction, 8)
 self.fc1 = nn.Linear(channels, hidden)
 self.fc2 = nn.Linear(hidden, channels)

 def forward(self, x: torch.Tensor) -> torch.Tensor:
 z = F.adaptive_avg_pool3d(x, 1).flatten(1)
 z = F.relu(self.fc1(z), inplace=True)
 z = torch.sigmoid(self.fc2(z)).unsqueeze(-1).unsqueeze(-1).unsqueeze(-1)
 return x * z


class AnatomyAwareExpertHead(nn.Module):
 def __init__(self, trunk_ch: int = 64, anat_ch: int = 16, region_vocab: int = 16, region_dim: int = 8, num_classes: int = 4):
 super().__init__()
 self.region_embed = nn.Embedding(region_vocab, region_dim)
 self.anat_proj = ConvNormAct(anat_ch, anat_ch)
 self.refine1 = ResidualBlock3D(trunk_ch + anat_ch + region_dim, 64)
 self.se = SEBlock3D(64)
 self.refine2 = ResidualBlock3D(64, 64)
 self.out = nn.Conv3d(64, num_classes, kernel_size=1)

 def forward(self, trunk: torch.Tensor, anat_feat: torch.Tensor, region: torch.Tensor) -> torch.Tensor:
 anat = self.anat_proj(anat_feat)
 if anat.shape[-3:] != trunk.shape[-3:]:
 anat = F.interpolate(anat, size=trunk.shape[-3:], mode='trilinear', align_corners=False)
 region_emb = self.region_embed(region.clamp_min(0))
 region_emb = region_emb.permute(0, 4, 1, 2, 3).contiguous().float()
 if region_emb.shape[-3:] != trunk.shape[-3:]:
 region_emb = F.interpolate(region_emb, size=trunk.shape[-3:], mode='nearest')
 x = torch.cat([trunk, anat, region_emb], dim=1)
 x = self.refine1(x)
 x = self.se(x)
 x = self.refine2(x)
 return self.out(x)


class FusionMLP(nn.Module):
 def __init__(self, input_dim: int, num_experts: int = 3):
 super().__init__()
 self.net = nn.Sequential(
 nn.Linear(input_dim, 128),
 nn.ReLU(inplace=True),
 nn.Dropout(0.1),
 nn.Linear(128, 64),
 nn.ReLU(inplace=True),
 nn.Linear(64, num_experts),
 )

 def forward(self, x: torch.Tensor) -> torch.Tensor:
 return self.net(x)


class ASWAFNet(nn.Module):
 def __init__(self, num_classes: int = 4, num_prior_channels: int = 9, region_vocab: int = 16):
 super().__init__()
 self.num_classes = num_classes
 self.mri_encoder = MRIEncoder3D(in_channels=4, base=32)
 self.anat_encoder = AnatomyEncoder3D(in_channels=num_prior_channels, base=16)

 self.d3 = DecoderBlock3D(320, 256, 256, anat_ch=128)
 self.d2 = DecoderBlock3D(256, 128, 128, anat_ch=64)
 self.d1 = DecoderBlock3D(128, 64, 64, anat_ch=32)
 self.d0 = DecoderBlock3D(64, 32, 32, anat_ch=16)

 self.shared_trunk = SharedTrunkHead(in_ch=32, out_ch=64)
 self.boundary_feature_proj = nn.Conv3d(16, 16, kernel_size=1)
 self.anat_feature_proj = nn.Conv3d(16, 16, kernel_size=1)

 self.context_expert = ContextExpertHead(trunk_ch=64, context_ch=128, num_classes=num_classes)
 self.boundary_expert = BoundaryExpertHead(trunk_ch=64, boundary_ch=16, edge_ch=16, num_classes=num_classes)
 self.anatomy_expert = AnatomyAwareExpertHead(trunk_ch=64, anat_ch=16, region_vocab=region_vocab, region_dim=8, num_classes=num_classes)

 self.fusion_mlp = FusionMLP(input_dim=320 + 160 + 3, num_experts=3)
 self.log_tau = nn.Parameter(torch.tensor(0.0))
 self.register_buffer('calibration_scores', torch.zeros(3))

 def set_calibration_scores(self, scores: torch.Tensor) -> None:
 if scores.shape != (3,):
 raise ValueError(f'Expected calibration scores shape (3,), got {scores.shape}')
 self.calibration_scores.copy_(scores.detach().float())

 def _entropy_summary(self, logits: torch.Tensor) -> torch.Tensor:
 probs = torch.softmax(logits, dim=1)
 entropy = -(probs * torch.log(probs.clamp_min(1e-8))).sum(dim=1)
 return entropy.mean(dim=(1, 2, 3), keepdim=False).unsqueeze(1)

 def forward(self, image: torch.Tensor, priors: torch.Tensor, region: torch.Tensor) -> Dict[str, torch.Tensor]:
 f0, f1, f2, f3, f4 = self.mri_encoder(image)
 g0, g1, g2, g3, g4 = self.anat_encoder(priors)

 d3 = self.d3(f4, f3, g3)
 d2 = self.d2(d3, f2, g2)
 d1 = self.d1(d2, f1, g1)
 d0 = self.d0(d1, f0, g0)

 trunk = self.shared_trunk(d0)
 boundary_feat = self.boundary_feature_proj(g0)
 anatomy_feat = self.anat_feature_proj(g0)

 logits_1 = self.context_expert(trunk, f2)
 logits_2 = self.boundary_expert(trunk, f0, boundary_feat)
 logits_3 = self.anatomy_expert(trunk, anatomy_feat, region)

 z_img = F.adaptive_avg_pool3d(f4, 1).flatten(1)
 z_anat = F.adaptive_avg_pool3d(g4, 1).flatten(1)
 u1 = self._entropy_summary(logits_1)
 u2 = self._entropy_summary(logits_2)
 u3 = self._entropy_summary(logits_3)
 fusion_in = torch.cat([z_img, z_anat, u1, u2, u3], dim=1)

 residual_scores = self.fusion_mlp(fusion_in)
 scores = residual_scores + self.calibration_scores.unsqueeze(0)
 tau = torch.exp(self.log_tau).clamp_min(1e-3)
 weights = torch.softmax(scores / tau, dim=1)

 fused_logits = (
 weights[:, 0].view(-1, 1, 1, 1, 1) * logits_1
 + weights[:, 1].view(-1, 1, 1, 1, 1) * logits_2
 + weights[:, 2].view(-1, 1, 1, 1, 1) * logits_3
 )

 return {
 'logits_fused': fused_logits,
 'logits_expert_1': logits_1,
 'logits_expert_2': logits_2,
 'logits_expert_3': logits_3,
 'weights': weights,
 }

In [ ]:
class AnatomyAwareExpertHead(nn.Module):
 """Anatomy expert with safe region-id clamping."""
 def __init__(self, trunk_ch: int = 64, anat_ch: int = 16, region_vocab: int = 16, region_dim: int = 8, num_classes: int = 4):
 super().__init__()
 self.region_embed = nn.Embedding(region_vocab, region_dim)
 self.anat_proj = ConvNormAct(anat_ch, anat_ch)
 self.refine1 = ResidualBlock3D(trunk_ch + anat_ch + region_dim, 64)
 self.se = SEBlock3D(64)
 self.refine2 = ResidualBlock3D(64, 64)
 self.out = nn.Conv3d(64, num_classes, kernel_size=1)

 def forward(self, trunk: torch.Tensor, anat_feat: torch.Tensor, region: torch.Tensor) -> torch.Tensor:
 anat = self.anat_proj(anat_feat)
 if anat.shape[-3:] != trunk.shape[-3:]:
 anat = F.interpolate(anat, size=trunk.shape[-3:], mode='trilinear', align_corners=False)

 max_region_id = self.region_embed.num_embeddings - 1
 region_safe = region.long().clamp(0, max_region_id)
 region_emb = self.region_embed(region_safe) # [B, D, H, W, region_dim]
 region_emb = region_emb.permute(0, 4, 1, 2, 3).contiguous().float()
 if region_emb.shape[-3:] != trunk.shape[-3:]:
 region_emb = F.interpolate(region_emb, size=trunk.shape[-3:], mode='nearest')

 x = torch.cat([trunk, anat, region_emb], dim=1)
 x = self.refine1(x)
 x = self.se(x)
 x = self.refine2(x)
 return self.out(x)


class RegionReliabilityRouter(nn.Module):
 """
 Efficient region-level anatomy-conditioned calibrated expert router.

 Inputs
 ------
 logits_1/logits_2/logits_3: [B, C, D, H, W]
 region: [B, D, H, W], integer region ids
 priors: [B, 9, D, H, W]
 z_img: [B, 320]
 z_anat: [B, 160]

 Outputs
 -------
 region_weights: [B, R, 3]
 router_scores: [B, R, 3]
 voxel_weights: [B, 3, D, H, W]
 region_fraction: [B, R]
 """
 def __init__(
 self,
 num_regions: int = 6,
 num_experts: int = 3,
 num_classes: int = 4,
 z_img_dim: int = 320,
 z_anat_dim: int = 160,
 region_dim: int = 8,
 global_dim: int = 16,
 hidden_dim: int = 32,
 lambda_calib: float = 1.0,
 ):
 super().__init__()
 self.num_regions = int(num_regions)
 self.num_experts = int(num_experts)
 self.num_classes = int(num_classes)
 self.lambda_calib = float(lambda_calib)

 self.region_embed = nn.Embedding(self.num_regions, region_dim)
 self.global_proj = nn.Sequential(
 nn.Linear(z_img_dim + z_anat_dim, global_dim),
 nn.ReLU(inplace=True),
 )

 # Per-region features:
 # entropy per expert -> 3
 # confidence per expert -> 3
 # region fraction -> 1
 # anatomy-boundary fraction -> 1
 # region embedding -> region_dim
 # compressed global context -> global_dim
 feature_dim = (2 * self.num_experts) + 2 + region_dim + global_dim

 self.score_mlp = nn.Sequential(
 nn.Linear(feature_dim, hidden_dim),
 nn.ReLU(inplace=True),
 nn.Linear(hidden_dim, self.num_experts),
 )

 # Start near uniform routing while allowing immediate gradient flow through router features.
 nn.init.normal_(self.score_mlp[-1].weight, mean=0.0, std=1e-3)
 nn.init.zeros_(self.score_mlp[-1].bias)

 self.log_tau = nn.Parameter(torch.tensor(0.0))
 self.register_buffer('calibration_scores', torch.zeros(self.num_experts, self.num_regions))

 def set_calibration_scores(self, scores: torch.Tensor) -> None:
 """
 Set calibration table C[e, r].

 Accepted shapes:
 - [3, R] preferred
 - [R, 3] transposed automatically
 - [3] old global calibration scores, broadcast to all regions
 """
 scores = scores.detach().float().to(self.calibration_scores.device)

 if scores.shape == (self.num_experts,):
 scores = scores.view(self.num_experts, 1).expand(self.num_experts, self.num_regions)
 elif scores.shape == (self.num_regions, self.num_experts):
 scores = scores.transpose(0, 1).contiguous()
 elif scores.shape != (self.num_experts, self.num_regions):
 raise ValueError(
 f'Expected calibration scores shape ({self.num_experts}, {self.num_regions}), '
 f'({self.num_regions}, {self.num_experts}), or ({self.num_experts},), got {tuple(scores.shape)}'
 )

 self.calibration_scores.copy_(scores)

 def get_calibration_scores(self) -> torch.Tensor:
 return self.calibration_scores.detach().clone()

 def _region_reduce_mean(
 self,
 values: torch.Tensor,
 region_flat: torch.Tensor,
 counts_safe: torch.Tensor,
 ) -> torch.Tensor:
 """
 values: [B, E, N]
 region_flat: [B, N]
 counts_safe: [B, R]
 returns: [B, E, R]
 """
 B, E, N = values.shape
 sums = values.new_zeros(B, E, self.num_regions)
 index = region_flat.unsqueeze(1).expand(B, E, N)
 sums.scatter_add_(dim=2, index=index, src=values)
 return sums / counts_safe.unsqueeze(1)

 def forward(
 self,
 logits_1: torch.Tensor,
 logits_2: torch.Tensor,
 logits_3: torch.Tensor,
 region: torch.Tensor,
 priors: torch.Tensor,
 z_img: torch.Tensor,
 z_anat: torch.Tensor,
 ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
 B, C, D, H, W = logits_1.shape
 E = self.num_experts
 R = self.num_regions
 N = D * H * W

 region = region.long().clamp(0, R - 1)
 region_flat = region.reshape(B, N)

 ones = torch.ones(B, N, device=region.device, dtype=torch.float32)
 counts = torch.zeros(B, R, device=region.device, dtype=torch.float32)
 counts.scatter_add_(dim=1, index=region_flat, src=ones)
 counts_safe = counts.clamp_min(1.0)
 region_fraction = counts / float(max(N, 1)) # [B, R]

 # Detach statistics so routing learns from expert reliability cues without adding large
 # uncertainty-statistic backprop graphs. Expert logits still get gradients through fusion.
 stats_logits_stack = torch.stack([logits_1, logits_2, logits_3], dim=1).detach().float() # [B, 3, C, D, H, W]
 probs = torch.softmax(stats_logits_stack, dim=2)

 confidence = probs.max(dim=2).values # [B, 3, D, H, W]
 entropy = -(probs * torch.log(probs.clamp_min(1e-8))).sum(dim=2) # [B, 3, D, H, W]
 entropy = entropy / math.log(max(C, 2))

 entropy_flat = entropy.reshape(B, E, N)
 confidence_flat = confidence.reshape(B, E, N)

 entropy_region = self._region_reduce_mean(entropy_flat, region_flat, counts_safe) # [B, 3, R]
 confidence_region = self._region_reduce_mean(confidence_flat, region_flat, counts_safe) # [B, 3, R]

 if priors is not None and priors.dim() == 5 and priors.size(1) > 8:
 boundary = priors[:, 8].float()
 if boundary.shape[-3:] != (D, H, W):
 boundary = F.interpolate(
 boundary.unsqueeze(1),
 size=(D, H, W),
 mode='trilinear',
 align_corners=False,
 ).squeeze(1)
 else:
 boundary = torch.zeros(B, D, H, W, device=region.device, dtype=torch.float32)

 boundary_flat = boundary.reshape(B, N)
 boundary_sum = torch.zeros(B, R, device=region.device, dtype=torch.float32)
 boundary_sum.scatter_add_(dim=1, index=region_flat, src=boundary_flat)
 boundary_fraction = boundary_sum / counts_safe # [B, R]

 entropy_region = entropy_region.permute(0, 2, 1).contiguous() # [B, R, 3]
 confidence_region = confidence_region.permute(0, 2, 1).contiguous() # [B, R, 3]
 region_fraction_feat = region_fraction.unsqueeze(-1) # [B, R, 1]
 boundary_fraction_feat = boundary_fraction.unsqueeze(-1) # [B, R, 1]

 region_ids = torch.arange(R, device=region.device, dtype=torch.long)
 region_emb = self.region_embed(region_ids).unsqueeze(0).expand(B, R, -1) # [B, R, region_dim]

 global_context = self.global_proj(torch.cat([z_img.float(), z_anat.float()], dim=1)) # [B, global_dim]
 global_context = global_context.unsqueeze(1).expand(B, R, -1) # [B, R, global_dim]

 router_features = torch.cat(
 [
 entropy_region,
 confidence_region,
 region_fraction_feat,
 boundary_fraction_feat,
 region_emb,
 global_context,
 ],
 dim=-1,
 )

 learned_scores = self.score_mlp(router_features) # [B, R, 3]
 calib = self.calibration_scores.transpose(0, 1).unsqueeze(0) # [1, R, 3]
 router_scores = learned_scores + self.lambda_calib * calib

 tau = torch.exp(self.log_tau).clamp_min(1e-3)
 region_weights = torch.softmax(router_scores / tau, dim=-1) # [B, R, 3]

 weights_by_expert = region_weights.permute(0, 2, 1).contiguous() # [B, 3, R]
 gather_index = region_flat.unsqueeze(1).expand(B, E, N) # [B, 3, N]
 voxel_weights = torch.gather(weights_by_expert, dim=2, index=gather_index)
 voxel_weights = voxel_weights.reshape(B, E, D, H, W) # [B, 3, D, H, W]

 return region_weights, router_scores, voxel_weights, region_fraction


class ACFNet(nn.Module):
 """
 acf_net:
 acf_net backbone/experts + efficient anatomy-conditioned calibrated region routing.
 """
 def __init__(
 self,
 num_classes: int = 4,
 num_prior_channels: int = 9,
 num_regions: int = 6,
 region_vocab: int = 16,
 router_hidden: int = 32,
 router_region_dim: int = 8,
 router_global_dim: int = 16,
 router_calibration_lambda: float = 1.0,
 debug_shapes: bool = False,
 return_voxel_weights: bool = False,
 ):
 super().__init__()
 self.model_name = 'acf_net'
 self.num_classes = int(num_classes)
 self.num_regions = int(num_regions)
 self.debug_shapes = bool(debug_shapes)
 self.return_voxel_weights = bool(return_voxel_weights)
 self._voxel_weights_returned_once = False

 self.mri_encoder = MRIEncoder3D(in_channels=4, base=32)
 self.anat_encoder = AnatomyEncoder3D(in_channels=num_prior_channels, base=16)

 self.d3 = DecoderBlock3D(320, 256, 256, anat_ch=128)
 self.d2 = DecoderBlock3D(256, 128, 128, anat_ch=64)
 self.d1 = DecoderBlock3D(128, 64, 64, anat_ch=32)
 self.d0 = DecoderBlock3D(64, 32, 32, anat_ch=16)

 self.shared_trunk = SharedTrunkHead(in_ch=32, out_ch=64)
 self.boundary_feature_proj = nn.Conv3d(16, 16, kernel_size=1)
 self.anat_feature_proj = nn.Conv3d(16, 16, kernel_size=1)

 safe_region_vocab = max(int(region_vocab), self.num_regions)

 self.context_expert = ContextExpertHead(
 trunk_ch=64,
 context_ch=128,
 num_classes=num_classes,
 )
 self.boundary_expert = BoundaryExpertHead(
 trunk_ch=64,
 boundary_ch=16,
 edge_ch=16,
 num_classes=num_classes,
 )
 self.anatomy_expert = AnatomyAwareExpertHead(
 trunk_ch=64,
 anat_ch=16,
 region_vocab=safe_region_vocab,
 region_dim=8,
 num_classes=num_classes,
 )

 self.router = RegionReliabilityRouter(
 num_regions=self.num_regions,
 num_experts=3,
 num_classes=num_classes,
 z_img_dim=320,
 z_anat_dim=160,
 region_dim=router_region_dim,
 global_dim=router_global_dim,
 hidden_dim=router_hidden,
 lambda_calib=router_calibration_lambda,
 )

 def set_calibration_scores(self, scores: torch.Tensor) -> None:
 self.router.set_calibration_scores(scores)

 def get_calibration_scores(self) -> torch.Tensor:
 return self.router.get_calibration_scores()

 def _prepare_region(self, region: torch.Tensor, target_shape: Tuple[int, int, int]) -> torch.Tensor:
 if region.dim() == 5:
 if region.size(1) == 1:
 region = region[:, 0]
 else:
 region = torch.argmax(region, dim=1)

 if region.shape[-3:] != target_shape:
 region = F.interpolate(
 region.unsqueeze(1).float(),
 size=target_shape,
 mode='nearest',
 ).squeeze(1)

 return region.long().clamp(0, self.num_regions - 1)

 def forward(self, image: torch.Tensor, priors: torch.Tensor, region: torch.Tensor) -> Dict[str, torch.Tensor]:
 region = self._prepare_region(region, image.shape[-3:])

 f0, f1, f2, f3, f4 = self.mri_encoder(image)
 g0, g1, g2, g3, g4 = self.anat_encoder(priors)

 d3 = self.d3(f4, f3, g3)
 d2 = self.d2(d3, f2, g2)
 d1 = self.d1(d2, f1, g1)
 d0 = self.d0(d1, f0, g0)

 trunk = self.shared_trunk(d0)
 boundary_feat = self.boundary_feature_proj(g0)
 anatomy_feat = self.anat_feature_proj(g0)

 logits_1 = self.context_expert(trunk, f2)
 logits_2 = self.boundary_expert(trunk, f0, boundary_feat)
 logits_3 = self.anatomy_expert(trunk, anatomy_feat, region)

 z_img = F.adaptive_avg_pool3d(f4, 1).flatten(1) # [B, 320]
 z_anat = F.adaptive_avg_pool3d(g4, 1).flatten(1) # [B, 160]

 region_weights, router_scores, voxel_weights, region_fraction = self.router(
 logits_1=logits_1,
 logits_2=logits_2,
 logits_3=logits_3,
 region=region,
 priors=priors,
 z_img=z_img,
 z_anat=z_anat,
 )

 logits_stack = torch.stack([logits_1, logits_2, logits_3], dim=1) # [B, 3, C, D, H, W]
 fused_logits = (voxel_weights.unsqueeze(2) * logits_stack).sum(dim=1) # [B, C, D, H, W]

 # Preserve existing key "weights" as an image-level summary [B, 3].
 image_weights = (region_weights * region_fraction.unsqueeze(-1)).sum(dim=1)

 outputs = {
 'logits_fused': fused_logits,
 'logits_expert_1': logits_1,
 'logits_expert_2': logits_2,
 'logits_expert_3': logits_3,
 'weights': image_weights,
 'region_weights': region_weights,
 'router_scores': router_scores,
 }

 include_voxel_weights = self.return_voxel_weights or (
 self.debug_shapes and not self._voxel_weights_returned_once
 )
 if include_voxel_weights:
 outputs['voxel_weights'] = voxel_weights.detach()
 self._voxel_weights_returned_once = True

 return outputs


In [ ]:
class SoftDiceLoss(nn.Module):
 def __init__(self, num_classes: int, smooth: float = 1e-5):
 super().__init__()
 self.num_classes = num_classes
 self.smooth = smooth

 def forward(self, logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
 probs = torch.softmax(logits, dim=1)
 target_oh = F.one_hot(target, num_classes=self.num_classes).permute(0, 4, 1, 2, 3).float()
 dims = (0, 2, 3, 4)
 intersection = (probs * target_oh).sum(dims)
 denom = probs.sum(dims) + target_oh.sum(dims)
 dice = (2.0 * intersection + self.smooth) / (denom + self.smooth)
 return 1.0 - dice[1:].mean()


def dice_per_class_from_logits(logits: torch.Tensor, target: torch.Tensor, num_classes: int) -> Dict[str, float]:
 pred = torch.argmax(logits, dim=1)
 metrics = {}
 for c in range(1, num_classes):
 pred_c = (pred == c).float()
 target_c = (target == c).float()
 intersection = (pred_c * target_c).sum().item()
 denom = pred_c.sum().item() + target_c.sum().item()
 dice = (2.0 * intersection + 1e-5) / (denom + 1e-5)
 metrics[f'dice_class_{c}'] = float(dice)
 fg_values = [metrics[f'dice_class_{c}'] for c in range(1, num_classes)]
 metrics['mean_dice'] = float(sum(fg_values) / len(fg_values))
 return metrics


def compute_total_loss(outputs: Dict[str, torch.Tensor], target: torch.Tensor, dice_loss: nn.Module) -> Tuple[torch.Tensor, Dict[str, float]]:
 ce = nn.CrossEntropyLoss()
 fused = outputs['logits_fused']
 e1 = outputs['logits_expert_1']
 e2 = outputs['logits_expert_2']
 e3 = outputs['logits_expert_3']

 loss_fused = dice_loss(fused, target) + ce(fused, target)
 loss_e1 = dice_loss(e1, target) + ce(e1, target)
 loss_e2 = dice_loss(e2, target) + ce(e2, target)
 loss_e3 = dice_loss(e3, target) + ce(e3, target)
 total = loss_fused + 0.3 * (loss_e1 + loss_e2 + loss_e3)

 parts = {
 'loss_total': float(total.detach().item()),
 'loss_fused': float(loss_fused.detach().item()),
 'loss_e1': float(loss_e1.detach().item()),
 'loss_e2': float(loss_e2.detach().item()),
 'loss_e3': float(loss_e3.detach().item()),
 }
 return total, parts

In [ ]:
class EarlyStopping:
 def __init__(self, patience: int, mode: str = 'max', min_delta: float = 1e-4):
 if mode not in {'max', 'min'}:
 raise ValueError("mode must be 'max' or 'min'")
 self.patience = patience
 self.mode = mode
 self.min_delta = min_delta
 self.best_score = None
 self.num_bad_epochs = 0
 self.should_stop = False

 def step(self, current: float) -> bool:
 if self.best_score is None:
 self.best_score = current
 return False

 improved = (
 current > self.best_score + self.min_delta
 if self.mode == 'max'
 else current < self.best_score - self.min_delta
 )

 if improved:
 self.best_score = current
 self.num_bad_epochs = 0
 else:
 self.num_bad_epochs += 1
 if self.num_bad_epochs >= self.patience:
 self.should_stop = True

 return self.should_stop

 def state_dict(self) -> Dict:
 return {
 'patience': self.patience,
 'mode': self.mode,
 'min_delta': self.min_delta,
 'best_score': self.best_score,
 'num_bad_epochs': self.num_bad_epochs,
 'should_stop': self.should_stop,
 }

 def load_state_dict(self, state: Dict) -> None:
 self.patience = state.get('patience', self.patience)
 self.mode = state.get('mode', self.mode)
 self.min_delta = state.get('min_delta', self.min_delta)
 self.best_score = state.get('best_score', self.best_score)
 self.num_bad_epochs = state.get('num_bad_epochs', self.num_bad_epochs)
 self.should_stop = state.get('should_stop', self.should_stop)


def capture_rng_state() -> Dict[str, Any]:
 state = {
 'torch_rng_state': torch.get_rng_state(),
 'numpy_rng_state': np.random.get_state(),
 'python_random_state': random.getstate(),
 }
 state['cuda_rng_state_all'] = torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
 return state


def restore_rng_state(state: Dict[str, Any]) -> None:
 try:
 torch_state = state.get('torch_rng_state', None)
 if torch_state is not None:
 torch.set_rng_state(torch_state.cpu())

 np_state = state.get('numpy_rng_state', None)
 if np_state is not None:
 np.random.set_state(np_state)

 py_state = state.get('python_random_state', None)
 if py_state is not None:
 random.setstate(py_state)

 cuda_state = state.get('cuda_rng_state_all', None)
 if torch.cuda.is_available() and cuda_state is not None:
 torch.cuda.set_rng_state_all(cuda_state)
 except Exception as exc:
 print(f"[Checkpoint warning] RNG state could not be fully restored: {repr(exc)}")


def _safe_torch_load(path: str, map_location: str = 'cpu') -> Dict:
 try:
 return torch.load(path, map_location=map_location, weights_only=False)
 except TypeError:
 return torch.load(path, map_location=map_location)


def inspect_state_dict_compatibility(model: nn.Module, loaded_state: Dict[str, torch.Tensor]) -> Dict[str, List]:
 current_state = model.state_dict()
 missing_keys = [k for k in current_state.keys() if k not in loaded_state]
 unexpected_keys = [k for k in loaded_state.keys() if k not in current_state]

 shape_mismatches = []
 for k in current_state.keys():
 if k in loaded_state:
 current_shape = tuple(current_state[k].shape)
 loaded_shape = tuple(loaded_state[k].shape)
 if current_shape != loaded_shape:
 shape_mismatches.append((k, loaded_shape, current_shape))

 return {
 'missing_keys': missing_keys,
 'unexpected_keys': unexpected_keys,
 'shape_mismatches': shape_mismatches,
 }


def format_mismatch_report(report: Dict[str, List], max_items: int = 20) -> str:
 lines = []
 missing = report.get('missing_keys', [])
 unexpected = report.get('unexpected_keys', [])
 mismatches = report.get('shape_mismatches', [])

 if missing:
 lines.append(f"Missing keys: {len(missing)}")
 for k in missing[:max_items]:
 lines.append(f" missing: {k}")
 if len(missing) > max_items:
 lines.append(f" ... {len(missing) - max_items} more missing keys")

 if unexpected:
 lines.append(f"Unexpected keys: {len(unexpected)}")
 for k in unexpected[:max_items]:
 lines.append(f" unexpected: {k}")
 if len(unexpected) > max_items:
 lines.append(f" ... {len(unexpected) - max_items} more unexpected keys")

 if mismatches:
 lines.append(f"Shape mismatches: {len(mismatches)}")
 for k, loaded_shape, current_shape in mismatches[:max_items]:
 lines.append(f" shape: {k}: checkpoint {loaded_shape} vs model {current_shape}")
 if len(mismatches) > max_items:
 lines.append(f" ... {len(mismatches) - max_items} more shape mismatches")

 return "\n".join(lines) if lines else "No state_dict incompatibilities detected."


def save_checkpoint(
 checkpoint_path: str,
 model: nn.Module,
 optimizer: torch.optim.Optimizer,
 scaler: Optional[torch.cuda.amp.GradScaler],
 epoch: int,
 best_metric: float,
 history: List[Dict],
 early_stopping: EarlyStopping,
 config: TrainConfig,
 scheduler: Optional[torch.optim.lr_scheduler._LRScheduler] = None,
) -> None:
 ensure_dir(str(Path(checkpoint_path).parent))
 state = {
 'epoch': int(epoch),
 'model_name': getattr(model, 'model_name', model.__class__.__name__),
 'model_state': model.state_dict(),
 'optimizer_state': optimizer.state_dict() if optimizer is not None else None,
 'scheduler_state': scheduler.state_dict() if scheduler is not None else None,
 'scaler_state': scaler.state_dict() if scaler is not None else None,
 'best_metric': float(best_metric),
 'history': history,
 'early_stopping': early_stopping.state_dict() if early_stopping is not None else None,
 'config': asdict(config),
 **capture_rng_state(),
 }
 torch.save(state, checkpoint_path)


def load_checkpoint(
 checkpoint_path: str,
 model: nn.Module,
 optimizer: Optional[torch.optim.Optimizer] = None,
 scaler: Optional[torch.cuda.amp.GradScaler] = None,
 scheduler: Optional[torch.optim.lr_scheduler._LRScheduler] = None,
 map_location: str = 'cpu',
 strict: bool = True,
) -> Dict:
 ckpt = _safe_torch_load(checkpoint_path, map_location=map_location)
 if 'model_state' not in ckpt:
 raise RuntimeError(f"Checkpoint does not contain 'model_state': {checkpoint_path}")

 report = inspect_state_dict_compatibility(model, ckpt['model_state'])
 has_incompatibility = bool(report['missing_keys'] or report['unexpected_keys'] or report['shape_mismatches'])
 if strict and has_incompatibility:
 raise RuntimeError(
 "Checkpoint is incompatible with the current architecture.\n"
 + format_mismatch_report(report)
 )

 if report['shape_mismatches']:
 # PyTorch cannot load mismatched tensor shapes even with strict=False.
 current_state = model.state_dict()
 filtered_state = {
 k: v for k, v in ckpt['model_state'].items()
 if k in current_state and tuple(v.shape) == tuple(current_state[k].shape)
 }
 model.load_state_dict(filtered_state, strict=False)
 else:
 model.load_state_dict(ckpt['model_state'], strict=strict)

 if optimizer is not None and ckpt.get('optimizer_state') is not None:
 try:
 optimizer.load_state_dict(ckpt['optimizer_state'])
 except Exception as exc:
 print(f"[Checkpoint warning] Optimizer state could not be loaded: {repr(exc)}")

 if scheduler is not None and ckpt.get('scheduler_state') is not None:
 try:
 scheduler.load_state_dict(ckpt['scheduler_state'])
 except Exception as exc:
 print(f"[Checkpoint warning] Scheduler state could not be loaded: {repr(exc)}")

 if scaler is not None and ckpt.get('scaler_state') is not None:
 try:
 scaler.load_state_dict(ckpt['scaler_state'])
 except Exception as exc:
 print(f"[Checkpoint warning] AMP scaler state could not be loaded: {repr(exc)}")

 restore_rng_state(ckpt)
 return ckpt


def latest_checkpoint_path(checkpoint_dir: str) -> Optional[str]:
 path = Path(checkpoint_dir)
 if not path.exists():
 return None

 latest = path / 'latest.pt'
 if latest.exists():
 return str(latest)

 ckpts = sorted(path.glob('epoch_*.pt'))
 if ckpts:
 return str(ckpts[-1])

 return None


def prune_old_epoch_checkpoints(checkpoint_dir: str, keep_last_k: int = 3) -> None:
 if keep_last_k is None or keep_last_k <= 0:
 return
 path = Path(checkpoint_dir)
 if not path.exists():
 return
 ckpts = sorted(path.glob('epoch_*.pt'))
 old = ckpts[:-keep_last_k]
 for p in old:
 try:
 p.unlink()
 except Exception as exc:
 print(f"[Checkpoint warning] Could not remove old checkpoint {p}: {repr(exc)}")


In [ ]:
@torch.no_grad()
def validate_one_epoch(
 model: nn.Module,
 loader: DataLoader,
 device: str,
 dice_loss: nn.Module,
 num_classes: int,
 epoch: Optional[int] = None,
 debug_shapes: bool = False,
) -> Dict[str, float]:
 model.eval()
 loss_meter = AverageMeter()
 dice_meter = AverageMeter()
 class_meters = {f'dice_class_{c}': AverageMeter() for c in range(1, num_classes)}

 desc = f"Val epoch {epoch}" if epoch is not None else "Validation"
 pbar = tqdm(loader, desc=desc, leave=False)

 for batch_idx, batch in enumerate(pbar):
 image = batch['image'].to(device, non_blocking=True)
 priors = batch['priors'].to(device, non_blocking=True)
 region = batch['region'].to(device, non_blocking=True)
 target = batch['seg'].to(device, non_blocking=True)

 outputs = model(image, priors, region)
 if debug_shapes and batch_idx == 0:
 print_debug_shapes_once(image, priors, region, outputs, prefix='[DEBUG_SHAPES validation first batch]')

 loss, _ = compute_total_loss(outputs, target, dice_loss)
 metrics = dice_per_class_from_logits(outputs['logits_fused'], target, num_classes)

 bsz = image.size(0)
 loss_meter.update(float(loss.item()), bsz)
 dice_meter.update(metrics['mean_dice'], bsz)
 for c in range(1, num_classes):
 class_meters[f'dice_class_{c}'].update(metrics[f'dice_class_{c}'], bsz)

 pbar.set_postfix({
 'loss': f'{loss_meter.avg:.4f}',
 'dice': f'{dice_meter.avg:.4f}',
 })

 result = {'val_loss': loss_meter.avg, 'val_mean_dice': dice_meter.avg}
 for c in range(1, num_classes):
 result[f'val_dice_class_{c}'] = class_meters[f'dice_class_{c}'].avg
 return result


def train_one_epoch(
 model: nn.Module,
 loader: DataLoader,
 optimizer: torch.optim.Optimizer,
 scaler: Optional[torch.cuda.amp.GradScaler],
 device: str,
 dice_loss: nn.Module,
 num_classes: int,
 amp: bool = True,
 grad_clip_norm: float = 12.0,
 epoch: Optional[int] = None,
 debug_shapes: bool = False,
) -> Dict[str, float]:
 model.train()
 loss_meter = AverageMeter()
 dice_meter = AverageMeter()

 desc = f"Train epoch {epoch}" if epoch is not None else "Training"
 pbar = tqdm(loader, desc=desc, leave=False)

 for batch_idx, batch in enumerate(pbar):
 image = batch['image'].to(device, non_blocking=True)
 priors = batch['priors'].to(device, non_blocking=True)
 region = batch['region'].to(device, non_blocking=True)
 target = batch['seg'].to(device, non_blocking=True)

 optimizer.zero_grad(set_to_none=True)
 with torch.cuda.amp.autocast(enabled=amp and device.startswith('cuda')):
 outputs = model(image, priors, region)
 loss, _ = compute_total_loss(outputs, target, dice_loss)

 if debug_shapes and batch_idx == 0:
 print_debug_shapes_once(image, priors, region, outputs, prefix='[DEBUG_SHAPES training first batch]')

 if scaler is not None and amp and device.startswith('cuda'):
 scaler.scale(loss).backward()
 scaler.unscale_(optimizer)
 torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)
 scaler.step(optimizer)
 scaler.update()
 else:
 loss.backward()
 torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)
 optimizer.step()

 if batch_idx == 0:
 print_gpu_memory('after first training batch')

 with torch.no_grad():
 metrics = dice_per_class_from_logits(outputs['logits_fused'], target, num_classes)
 bsz = image.size(0)
 loss_meter.update(float(loss.item()), bsz)
 dice_meter.update(metrics['mean_dice'], bsz)

 pbar.set_postfix({
 'loss': f'{loss_meter.avg:.4f}',
 'dice': f'{dice_meter.avg:.4f}',
 })

 return {'train_loss': loss_meter.avg, 'train_mean_dice': dice_meter.avg}


In [ ]:
def initialize_zero_region_calibration(model: nn.Module, cfg: TrainConfig) -> torch.Tensor:
 """
 acf_net starts with a zero region calibration table C[e, r].
 No extra validation/calibration pass is performed at startup.
 """
 scores = torch.zeros(3, cfg.num_regions, dtype=torch.float32, device=cfg.device)
 if hasattr(model, 'set_calibration_scores'):
 model.set_calibration_scores(scores)
 print(f"[Calibration] Initialized zero region calibration table with shape {tuple(scores.shape)}")
 return scores


In [ ]:
def build_dataloaders(cfg: TrainConfig) -> Tuple[DataLoader, DataLoader, Optional[DataLoader], Dict[str, List[Dict]]]:
 print("[Data] Loading or creating acf_net manifest/splits...")
 print(f"[Data] Manifest path: {cfg.manifest_path}")
 print(f"[Data] Split path: {cfg.split_path}")

 cases = load_or_create_manifest(cfg)
 splits = load_or_create_splits(cfg, cases)

 print("[Data] Building datasets...")
 train_ds = BratsFastSurferDataset(cfg=cfg, samples=splits['train'], training=True)
 val_ds = BratsFastSurferDataset(cfg=cfg, samples=splits['val'], training=False)
 test_ds = BratsFastSurferDataset(cfg=cfg, samples=splits['test'], training=False) if len(splits['test']) > 0 else None

 print(f"[Data] train cases: {len(train_ds)}")
 print(f"[Data] val cases: {len(val_ds)}")
 print(f"[Data] test cases: {len(test_ds) if test_ds is not None else 0}")
 print(f"[Data] batch_size={cfg.batch_size}, num_workers={cfg.num_workers}, patch_size={cfg.patch_size}")
 print(f"[Data] acf_net prior cache dir: {cfg.cache_dir}")

 print("[Data] Creating DataLoaders...")
 train_loader = DataLoader(
 train_ds,
 batch_size=cfg.batch_size,
 shuffle=True,
 num_workers=cfg.num_workers,
 pin_memory=cfg.pin_memory,
 persistent_workers=cfg.persistent_workers if cfg.num_workers > 0 else False,
 prefetch_factor=cfg.prefetch_factor if cfg.num_workers > 0 else None,
 )
 val_loader = DataLoader(
 val_ds,
 batch_size=1,
 shuffle=False,
 num_workers=cfg.num_workers,
 pin_memory=True,
 persistent_workers=cfg.num_workers > 0,
 )
 test_loader = None
 if test_ds is not None:
 test_loader = DataLoader(
 test_ds,
 batch_size=1,
 shuffle=False,
 num_workers=cfg.num_workers,
 pin_memory=True,
 persistent_workers=cfg.num_workers > 0,
 )

 print("[Data] DataLoaders ready.")
 return train_loader, val_loader, test_loader, splits


In [ ]:
def verify_dataset_roots(cfg: TrainConfig) -> None:
 print('Checking dataset roots...')
 print(f'BRATS root: {cfg.brats_root}')
 print(f'FastSurfer root: {cfg.fastsurfer_root}')

 brats_exists = os.path.isdir(cfg.brats_root)
 fs_exists = os.path.isdir(cfg.fastsurfer_root)
 print(f'BRATS root exists: {brats_exists}')
 print(f'FastSurfer root exists: {fs_exists}')

 if not brats_exists or not fs_exists:
 raise FileNotFoundError(
 'One or both dataset roots do not exist. These defaults assume Colab Drive is mounted under FILL IN WITH DIRECTORY FOR GOOGLE DRIVE ROOT.'
 )


def print_matched_case_summary(cfg: TrainConfig, cases: Optional[List[Dict]] = None) -> List[Dict]:
 if cases is None:
 cases = load_or_create_manifest(cfg)
 print(f'Matched cases found: {len(cases)}')
 print(f"First 3 matched case IDs: {[c['id'] for c in cases[:3]]}")
 return cases


def inspect_sample_case_shapes(case: Dict) -> Dict[str, np.ndarray]:
 print(f"Inspecting raw sample case: {case['id']}")
 vols = {
 't1': load_volume(case['mri']['t1'], dtype=np.float32),
 't1ce': load_volume(case['mri']['t1ce'], dtype=np.float32),
 't2': load_volume(case['mri']['t2'], dtype=np.float32),
 'flair': load_volume(case['mri']['flair'], dtype=np.float32),
 'seg': load_volume(case['seg'], dtype=np.int32),
 'dkt': load_volume(case['fastsurfer']['dkt'], dtype=np.int32),
 'aseg': load_volume(case['fastsurfer']['aseg'], dtype=np.int32),
 'mask': load_volume(case['fastsurfer']['mask'], dtype=np.float32),
 }
 for name, arr in vols.items():
 print(f'{name:>6} shape: {arr.shape}')
 return vols


def inspect_sample_priors(cfg: TrainConfig, case: Dict) -> Tuple[np.ndarray, np.ndarray]:
 print(f"Building anatomy priors for sample case: {case['id']}")
 ref_t1 = load_volume(case['mri']['t1'], dtype=np.float32)
 priors, region = derive_anatomy_priors_from_fastsurfer(case, verbose=False)

 print(f'MRI shape: {ref_t1.shape}')
 print(f'resampled priors shape: {priors.shape}')
 print(f'resampled region shape: {region.shape}')
 print(f'unique region values: {np.unique(region).tolist()}')
 print(f'wm_dist min/max: {float(priors[6].min()):.6f} / {float(priors[6].max()):.6f}')
 print(f'vent_dist min/max: {float(priors[7].min()):.6f} / {float(priors[7].max()):.6f}')
 return priors, region


def plot_sample_case_slices(case: Dict, priors: Optional[np.ndarray] = None, region: Optional[np.ndarray] = None) -> None:
 t1ce = load_volume(case['mri']['t1ce'], dtype=np.float32)
 seg = remap_brats_labels(load_volume(case['seg'], dtype=np.int32))
 if priors is None or region is None:
 priors, region = derive_anatomy_priors_from_fastsurfer(case)

 z = t1ce.shape[0] // 2
 wm = priors[0]
 vent = priors[3]
 cortex = priors[5]
 anat_boundary = priors[8]

 items = [
 ('t1ce', t1ce[z], 'gray'),
 ('seg', seg[z], 'viridis'),
 ('wm prior', wm[z], 'gray'),
 ('ventricle prior', vent[z], 'gray'),
 ('cortex prior', cortex[z], 'gray'),
 ('anat_boundary', anat_boundary[z], 'gray'),
 ('region map', region[z], 'tab20'),
 ]

 fig, axes = plt.subplots(1, len(items), figsize=(4 * len(items), 4))
 if len(items) == 1:
 axes = [axes]
 for ax, (title, img, cmap) in zip(axes, items):
 ax.imshow(np.rot90(img), cmap=cmap)
 ax.set_title(title)
 ax.axis('off')
 plt.tight_layout()
 plt.show()


def sanity_check_dataloader(cfg: TrainConfig, splits: Optional[Dict[str, List[Dict]]] = None) -> None:
 if splits is None:
 cases = load_or_create_manifest(cfg)
 splits = load_or_create_splits(cfg, cases)
 ds = BratsFastSurferDataset(cfg=cfg, samples=splits['train'], training=True)
 loader = DataLoader(ds, batch_size=min(cfg.batch_size, 1), shuffle=False, num_workers=0, pin_memory=False)
 batch = next(iter(loader))
 print('Loaded 1 batch from DataLoader')
 print(f"image tensor shape: {tuple(batch['image'].shape)}")
 print(f"priors tensor shape: {tuple(batch['priors'].shape)}")
 print(f"region tensor shape: {tuple(batch['region'].shape)}")
 print(f"seg tensor shape: {tuple(batch['seg'].shape)}")
 print(f"case ids: {batch['id']}")


def run_pretraining_verification(cfg: TrainConfig) -> Dict[str, List[Dict]]:
 verify_dataset_roots(cfg)
 cases = load_or_create_manifest(cfg)
 print_matched_case_summary(cfg, cases)
 if len(cases) == 0:
 raise RuntimeError('No matched cases found after scanning BRATS and FastSurfer roots.')

 sample_case = cases[0]
 inspect_sample_case_shapes(sample_case)
 priors, region = inspect_sample_priors(cfg, sample_case)
 plot_sample_case_slices(sample_case, priors=priors, region=region)

 splits = load_or_create_splits(cfg, cases)
 sanity_check_dataloader(cfg, splits)
 return splits

In [ ]:
def fit(cfg: TrainConfig) -> None:
 mount_drive_in_colab()

 print("=" * 80)
 print("acf_net training startup")
 print("=" * 80)
 print(f"Device: {cfg.device}")
 if torch.cuda.is_available():
 print(f"GPU: {torch.cuda.get_device_name(0)}")
 else:
 print("GPU: not available, using CPU")
 print(f"Smoke test mode: {cfg.smoke_test}")
 if cfg.smoke_test:
 print(f"Smoke test epochs: {cfg.smoke_test_epochs}")

 print("\n[Isolation] This acf_net run uses separate output files for this experiment.")
 print(f"[Isolation] work_dir: {cfg.work_dir}")
 print(f"[Isolation] cache_dir: {cfg.cache_dir}")
 print(f"[Isolation] checkpoint_dir: {cfg.checkpoint_dir}")
 print(f"[Isolation] log_dir: {cfg.log_dir}")

 set_seed(cfg.seed)

 ensure_dir(cfg.work_dir)
 ensure_dir(cfg.cache_dir)
 ensure_dir(cfg.checkpoint_dir)
 ensure_dir(cfg.log_dir)
 save_json(asdict(cfg), os.path.join(cfg.log_dir, "train_config.json"))

 print("\n[Startup] Building dataloaders...")
 train_loader, val_loader, _, splits = build_dataloaders(cfg)
 print(f"[Startup] Split sizes: { {k: len(v) for k, v in splits.items()} }")

 if cfg.cache_priors and cfg.precompute_all_priors_before_training:
 print("\n[Startup] Precomputing cached priors for all splits...")
 all_cases = splits["train"] + splits["val"] + splits["test"]
 for case in tqdm(all_cases, desc='Precomputing acf_net priors'):
 load_or_build_priors(cfg, case)

 print("\n[Startup] Initializing acf_net model...")
 model = ACFNet(
 num_classes=cfg.num_classes,
 num_prior_channels=9,
 num_regions=cfg.num_regions,
 region_vocab=cfg.region_vocab,
 router_hidden=cfg.router_hidden,
 router_region_dim=cfg.router_region_dim,
 router_global_dim=cfg.router_global_dim,
 router_calibration_lambda=cfg.router_calibration_lambda,
 debug_shapes=cfg.DEBUG_SHAPES,
 return_voxel_weights=False,
 ).to(cfg.device)

 print_parameter_summary(
 model,
 model_name='acf_net',
 baseline_trainable_m=30.43,
 print_modules=True,
 )
 print_gpu_memory('after model creation')

 optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
 scaler = torch.cuda.amp.GradScaler(enabled=cfg.amp and cfg.device.startswith("cuda"))
 scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.max_epochs)
 dice_loss = SoftDiceLoss(num_classes=cfg.num_classes)
 early_stopping = EarlyStopping(
 patience=cfg.patience,
 mode=cfg.monitor_mode,
 min_delta=cfg.min_delta,
 )

 # V1: zero calibration table. No expensive validation/calibration startup pass.
 initialize_zero_region_calibration(model, cfg)

 start_epoch = 0
 best_metric = -math.inf if cfg.monitor_mode == "max" else math.inf
 history: List[Dict] = []

 resume_path = cfg.resume_path
 if resume_path is None and cfg.AUTO_RESUME:
 resume_path = latest_checkpoint_path(cfg.checkpoint_dir)

 if resume_path is not None and os.path.exists(resume_path):
 print(f"\n[Auto-resume] Candidate checkpoint found: {resume_path}")
 try:
 ckpt = load_checkpoint(
 resume_path,
 model,
 optimizer=optimizer,
 scaler=scaler,
 scheduler=scheduler,
 map_location=cfg.device,
 strict=True,
 )
 completed_epoch = int(ckpt["epoch"])
 start_epoch = completed_epoch + 1
 best_metric = float(ckpt.get("best_metric", best_metric))
 history = list(ckpt.get("history", []))
 if ckpt.get("early_stopping") is not None:
 early_stopping.load_state_dict(ckpt["early_stopping"])

 print("[Auto-resume] Resume successful.")
 print(f"[Auto-resume] checkpoint path: {resume_path}")
 print(f"[Auto-resume] resumed epoch: {completed_epoch}")
 print(f"[Auto-resume] completed epochs: {len(history)}")
 print(f"[Auto-resume] best metric so far: {best_metric:.6f}")
 print(f"[Auto-resume] next epoch to run: {start_epoch}")
 except RuntimeError as exc:
 print("\n[Auto-resume] Checkpoint exists but is incompatible with the current acf_net architecture.")
 print(str(exc))
 if cfg.resume_path is not None:
 raise
 print("[Auto-resume] Starting from scratch. The incompatible checkpoint was not modified.")
 else:
 print("\n[Auto-resume] No valid latest checkpoint found. Training is starting from scratch.")

 history_path = os.path.join(cfg.log_dir, "history.json")
 effective_max_epochs = cfg.smoke_test_epochs if cfg.smoke_test else cfg.max_epochs

 print("\n[Startup] Starting training loop...")

 for epoch in range(start_epoch, effective_max_epochs):
 epoch_start = time.time()

 train_metrics = train_one_epoch(
 model,
 train_loader,
 optimizer,
 scaler,
 cfg.device,
 dice_loss,
 cfg.num_classes,
 amp=cfg.amp,
 grad_clip_norm=cfg.grad_clip_norm,
 epoch=epoch,
 debug_shapes=cfg.DEBUG_SHAPES and epoch == start_epoch,
 )

 val_metrics = validate_one_epoch(
 model,
 val_loader,
 cfg.device,
 dice_loss,
 cfg.num_classes,
 epoch=epoch,
 debug_shapes=False,
 )

 scheduler.step()

 epoch_time = time.time() - epoch_start
 row = {
 "epoch": epoch,
 "lr": optimizer.param_groups[0]["lr"],
 "time_sec": epoch_time,
 **train_metrics,
 **val_metrics,
 }
 history.append(row)
 save_json({"history": history}, history_path)

 current_metric = row[cfg.monitor_metric]
 improved = (
 current_metric > best_metric + cfg.min_delta
 if cfg.monitor_mode == "max"
 else current_metric < best_metric - cfg.min_delta
 )
 if improved:
 best_metric = float(current_metric)

 stop = early_stopping.step(float(current_metric))

 if improved:
 best_path = os.path.join(cfg.checkpoint_dir, "best_model.pt")
 save_checkpoint(
 best_path,
 model,
 optimizer,
 scaler,
 epoch,
 best_metric,
 history,
 early_stopping,
 cfg,
 scheduler=scheduler,
 )
 print(f"[Checkpoint] Saved best model: {best_path}")

 save_epoch_file = cfg.SAVE_EVERY_EPOCH or (cfg.save_every > 0 and ((epoch + 1) % cfg.save_every == 0))
 if save_epoch_file and not cfg.SAVE_BEST_ONLY:
 ckpt_path = os.path.join(cfg.checkpoint_dir, f"epoch_{epoch:04d}.pt")
 save_checkpoint(
 ckpt_path,
 model,
 optimizer,
 scaler,
 epoch,
 best_metric,
 history,
 early_stopping,
 cfg,
 scheduler=scheduler,
 )
 prune_old_epoch_checkpoints(cfg.checkpoint_dir, keep_last_k=cfg.keep_last_k)

 latest_path = os.path.join(cfg.checkpoint_dir, "latest.pt")
 save_checkpoint(
 latest_path,
 model,
 optimizer,
 scaler,
 epoch,
 best_metric,
 history,
 early_stopping,
 cfg,
 scheduler=scheduler,
 )

 print(
 f"Epoch {epoch:03d} | "
 f"train_loss={row['train_loss']:.4f} | "
 f"train_dice={row['train_mean_dice']:.4f} | "
 f"val_loss={row['val_loss']:.4f} | "
 f"val_mean_dice={row['val_mean_dice']:.4f} | "
 f"best={best_metric:.4f} | "
 f"lr={row['lr']:.2e} | "
 f"time={epoch_time:.1f}s"
 )

 if stop:
 print(f"Early stopping triggered at epoch {epoch}")
 break

 print("Training finished.")


In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("Device count:", torch.cuda.device_count())
if torch.cuda.is_available():
 print("Current device:", torch.cuda.current_device())
 print("GPU name:", torch.cuda.get_device_name(0))

In [ ]:
# Optional parameter sanity check cell for acf_net.
try:
 cfg
except NameError:
 cfg = TrainConfig()

model = ACFNet(
 num_classes=cfg.num_classes,
 num_prior_channels=9,
 num_regions=cfg.num_regions,
 region_vocab=cfg.region_vocab,
 router_hidden=cfg.router_hidden,
 router_region_dim=cfg.router_region_dim,
 router_global_dim=cfg.router_global_dim,
 router_calibration_lambda=cfg.router_calibration_lambda,
 debug_shapes=cfg.DEBUG_SHAPES,
).to(cfg.device)

print_parameter_summary(model, model_name='acf_net', baseline_trainable_m=30.43, print_modules=True)
print_gpu_memory('after optional acf_net model sanity creation')


In [ ]:
# Optional isolated checkpoint path sanity check.
try:
 cfg
except NameError:
 cfg = TrainConfig()

print("acf_net work_dir:", cfg.work_dir)
print("acf_net cache_dir:", cfg.cache_dir)
print("acf_net checkpoint_dir:", cfg.checkpoint_dir)
print("acf_net latest checkpoint candidate:", latest_checkpoint_path(cfg.checkpoint_dir))


In [ ]:
from google.colab import drive
drive.mount('FILL IN WITH DIRECTORY FOR COLAB DRIVE MOUNT', force_remount=True)

ABOVE IS V1 prelim blocks below are new v2. run all cells.

In [ ]:
from dataclasses import dataclass
from typing import Tuple, Optional, Dict, List
import os
import math
import time
import copy
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

@dataclass
class TrainConfigV2(TrainConfig):
 # Identity
 model_name: str = "acf_net"
 architecture_name: str = "acf_net-V2"

 # Fully isolated V2 paths
 work_dir: str = "FILL IN WITH DIRECTORY FOR ACF NET BRATS RUNS"
 checkpoint_dir: str = "FILL IN WITH DIRECTORY FOR MODEL CHECKPOINTS_v2"
 log_dir: str = "FILL IN WITH DIRECTORY FOR TRAINING LOGS_v2"
 results_dir: str = "FILL IN WITH DIRECTORY FOR EVALUATION RESULTS_v2"

 # Safe default: reuse split/manifest from acf_net run, but use a V2 prior cache.
 manifest_path: str = "FILL IN WITH DIRECTORY FOR RUN MANIFEST/manifest.json"
 split_path: str = "FILL IN WITH DIRECTORY FOR TRAIN VALIDATION TEST SPLITS/splits.json"
 cache_dir: str = "FILL IN WITH DIRECTORY FOR VALIDATION PRIOR CACHE_v2"

 # Training data roots
 brats_root: str = "FILL IN WITH DIRECTORY FOR BRATS GLI TRAINING DATA"
 fastsurfer_root: str = "FILL IN WITH DIRECTORY FOR FASTSURFER LABELED TRAINING OUTPUTS"

 # V2 data/sampling
 patch_size: Tuple[int, int, int] = (96, 96, 96)
 batch_size: int = 1
 use_patch_sampling: bool = True
 tumor_patch_prob: float = 0.70
 et_patch_prob: float = 0.30
 max_sampling_tries: int = 20
 min_foreground_voxels: int = 8

 # Colab loader settings
 num_workers: int = 2
 pin_memory: bool = True
 persistent_workers: bool = True
 prefetch_factor: int = 2

 # V2 loss
 use_region_loss: bool = True
 region_loss_weight: float = 0.35
 region_loss_wt_weight: float = 1.0
 region_loss_tc_weight: float = 1.0
 region_loss_et_weight: float = 1.35

 # Training strategy
 lr: float = 1e-4
 weight_decay: float = 1e-5
 max_epochs: int = 120
 patience: int = 22
 min_delta: float = 1e-4
 monitor_mode: str = "max"
 monitor_metric: str = "val_mean_region_dice"
 amp: bool = True
 grad_clip_norm: float = 12.0

 # Validation/checkpointing
 val_every: int = 1
 save_every: int = 5
 keep_last_k: int = 4
 SAVE_EVERY_EPOCH: bool = False
 SAVE_BEST_ONLY: bool = False
 AUTO_RESUME: bool = True
 DEBUG_SHAPES: bool = True

 # LR scheduling
 lr_plateau_patience: int = 5
 lr_plateau_factor: float = 0.5
 min_lr: float = 1e-6

 # Expensive validation metrics
 compute_val_hd95_every: int = 5

 # Device
 device: str = "cuda" if torch.cuda.is_available() else "cpu"


def prepare_v2_dirs(cfg: TrainConfigV2):
 for path in [cfg.checkpoint_dir, cfg.log_dir, cfg.results_dir, cfg.cache_dir]:
 ensure_dir(path)

 print("=" * 80)
 print("acf_net isolated run directories")
 print("=" * 80)
 print(f"checkpoint_dir: {cfg.checkpoint_dir}")
 print(f"log_dir: {cfg.log_dir}")
 print(f"results_dir: {cfg.results_dir}")
 print(f"cache_dir: {cfg.cache_dir}")

In [ ]:
class BratsFastSurferDatasetV2(BratsFastSurferDataset):
 """
 V2 dataset with tumor-aware and ET-aware patch sampling.

 Label convention after remap_brats_labels:
 0 = background
 1 = tumor core / non-enhancing / necrotic
 2 = edema
 3 = enhancing tumor

 WT = {1,2,3}
 TC = {1,3}
 ET = {3}
 """

 def _start_from_center(
 self,
 center: Tuple[int, int, int],
 shape: Tuple[int, int, int],
 target: Tuple[int, int, int],
 ) -> Tuple[int, int, int]:
 starts = []
 for c, dim, tgt in zip(center, shape, target):
 if dim <= tgt:
 starts.append(0)
 else:
 start = int(c) - tgt // 2
 start = max(0, min(start, dim - tgt))
 starts.append(start)
 return tuple(starts)

 def _sample_tumor_aware_start(
 self,
 seg: np.ndarray,
 target: Tuple[int, int, int],
 ) -> Tuple[int, int, int]:
 shape = seg.shape[-3:]

 foreground_coords = np.argwhere(seg > 0)
 et_coords = np.argwhere(seg == 3)

 use_et = (
 len(et_coords) > 0
 and random.random() < float(getattr(self.cfg, "et_patch_prob", 0.30))
 )

 use_tumor = (
 len(foreground_coords) > 0
 and random.random() < float(getattr(self.cfg, "tumor_patch_prob", 0.70))
 )

 if use_et:
 center = et_coords[random.randint(0, len(et_coords) - 1)]
 return self._start_from_center(tuple(center), shape, target)

 if use_tumor:
 center = foreground_coords[random.randint(0, len(foreground_coords) - 1)]
 return self._start_from_center(tuple(center), shape, target)

 return self._sample_start(shape, target)

 def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
 sample = self.samples[idx]

 t1 = zscore_nonzero(load_volume(sample["mri"]["t1"], dtype=np.float32))
 t1ce = zscore_nonzero(load_volume(sample["mri"]["t1ce"], dtype=np.float32))
 t2 = zscore_nonzero(load_volume(sample["mri"]["t2"], dtype=np.float32))
 flair = zscore_nonzero(load_volume(sample["mri"]["flair"], dtype=np.float32))
 seg = remap_brats_labels(load_volume(sample["seg"], dtype=np.int32)).astype(np.int64)

 x = np.stack([t1, t1ce, t2, flair], axis=0).astype(np.float32, copy=False)

 priors, region = load_or_build_priors(self.cfg, sample)

 x = self._pad_if_needed(x, self.cfg.patch_size, value=0)
 priors = self._pad_if_needed(priors, self.cfg.patch_size, value=0)
 seg = self._pad_if_needed(seg, self.cfg.patch_size, value=0)
 region = self._pad_if_needed(region, self.cfg.patch_size, value=0)

 if self.cfg.use_patch_sampling and self.training:
 start = self._sample_tumor_aware_start(seg, self.cfg.patch_size)

 # Optional safety: retry if patch has too little foreground.
 max_tries = int(getattr(self.cfg, "max_sampling_tries", 20))
 min_fg = int(getattr(self.cfg, "min_foreground_voxels", 8))

 for _ in range(max_tries):
 candidate_seg = self._random_crop(seg, start, self.cfg.patch_size)
 if int((candidate_seg > 0).sum()) >= min_fg:
 break
 start = self._sample_tumor_aware_start(seg, self.cfg.patch_size)

 x = self._random_crop(x, start, self.cfg.patch_size)
 priors = self._random_crop(priors, start, self.cfg.patch_size)
 seg = self._random_crop(seg, start, self.cfg.patch_size)
 region = self._random_crop(region, start, self.cfg.patch_size)
 else:
 x = self._center_crop(x, self.cfg.patch_size)
 priors = self._center_crop(priors, self.cfg.patch_size)
 seg = self._center_crop(seg, self.cfg.patch_size)
 region = self._center_crop(region, self.cfg.patch_size)

 return {
 "id": sample["id"],
 "image": torch.from_numpy(x).float(),
 "priors": torch.from_numpy(priors).float(),
 "region": torch.from_numpy(region).long(),
 "seg": torch.from_numpy(seg).long(),
 }

In [ ]:
BRATS_REGION_DEFS = {
 "wt": (1, 2, 3),
 "tc": (1, 3),
 "et": (3,),
}


def region_mask_from_probs(probs: torch.Tensor, labels: Tuple[int, ...]) -> torch.Tensor:
 return probs[:, labels].sum(dim=1)


def region_mask_from_target(target: torch.Tensor, labels: Tuple[int, ...]) -> torch.Tensor:
 mask = torch.zeros_like(target, dtype=torch.float32)
 for label in labels:
 mask = torch.maximum(mask, (target == label).float())
 return mask


class BraTSRegionDiceLoss(nn.Module):
 def __init__(
 self,
 wt_weight: float = 1.0,
 tc_weight: float = 1.0,
 et_weight: float = 1.35,
 smooth: float = 1e-5,
 ):
 super().__init__()
 self.weights = {
 "wt": float(wt_weight),
 "tc": float(tc_weight),
 "et": float(et_weight),
 }
 self.smooth = float(smooth)

 def forward(self, logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
 probs = torch.softmax(logits, dim=1)

 losses = []
 weights = []

 for name, labels in BRATS_REGION_DEFS.items():
 pred_region = region_mask_from_probs(probs, labels)
 target_region = region_mask_from_target(target, labels)

 dims = (0, 1, 2, 3)
 intersection = (pred_region * target_region).sum(dims)
 denom = pred_region.sum(dims) + target_region.sum(dims)
 dice = (2.0 * intersection + self.smooth) / (denom + self.smooth)

 losses.append(1.0 - dice)
 weights.append(self.weights[name])

 loss = sum(w * l for w, l in zip(weights, losses)) / max(sum(weights), 1e-8)
 return loss


def brats_region_dice_from_logits(logits: torch.Tensor, target: torch.Tensor) -> Dict[str, float]:
 pred = torch.argmax(logits, dim=1)
 out = {}

 vals = []
 for name, labels in BRATS_REGION_DEFS.items():
 pred_mask = torch.zeros_like(pred, dtype=torch.float32)
 target_mask = torch.zeros_like(target, dtype=torch.float32)

 for label in labels:
 pred_mask = torch.maximum(pred_mask, (pred == label).float())
 target_mask = torch.maximum(target_mask, (target == label).float())

 intersection = (pred_mask * target_mask).sum().item()
 denom = pred_mask.sum().item() + target_mask.sum().item()

 if denom == 0:
 dice = 1.0
 else:
 dice = (2.0 * intersection + 1e-5) / (denom + 1e-5)

 out[f"dice_{name}"] = float(dice)
 vals.append(float(dice))

 out["mean_region_dice"] = float(sum(vals) / len(vals))
 return out


def compute_total_loss_v2(
 outputs: Dict[str, torch.Tensor],
 target: torch.Tensor,
 dice_loss: nn.Module,
 region_dice_loss: nn.Module,
 cfg: TrainConfigV2,
) -> Tuple[torch.Tensor, Dict[str, float]]:
 ce = nn.CrossEntropyLoss()

 fused = outputs["logits_fused"]
 e1 = outputs["logits_expert_1"]
 e2 = outputs["logits_expert_2"]
 e3 = outputs["logits_expert_3"]

 loss_fused = dice_loss(fused, target) + ce(fused, target)
 loss_e1 = dice_loss(e1, target) + ce(e1, target)
 loss_e2 = dice_loss(e2, target) + ce(e2, target)
 loss_e3 = dice_loss(e3, target) + ce(e3, target)

 base_total = loss_fused + 0.3 * (loss_e1 + loss_e2 + loss_e3)

 if cfg.use_region_loss:
 region_loss = region_dice_loss(fused, target)
 total = base_total + float(cfg.region_loss_weight) * region_loss
 else:
 region_loss = torch.zeros((), device=target.device, dtype=base_total.dtype)
 total = base_total

 parts = {
 "loss_total": float(total.detach().item()),
 "loss_base": float(base_total.detach().item()),
 "loss_region": float(region_loss.detach().item()),
 "loss_fused": float(loss_fused.detach().item()),
 "loss_e1": float(loss_e1.detach().item()),
 "loss_e2": float(loss_e2.detach().item()),
 "loss_e3": float(loss_e3.detach().item()),
 }

 return total, parts

In [ ]:
class ACRSWAFNetV2(ACFNet):
 """
 V2 keeps the efficient V1 acf_net architecture.
 The main V2 changes are training-side:
 - BraTS-region-aware loss
 - tumor/ET-aware patch sampling
 - validation-selected checkpointing on val_mean_region_dice
 - improved LR schedule
 """
 def __init__(self, *args, **kwargs):
 super().__init__(*args, **kwargs)
 self.model_name = "acf_net"
 self.architecture_name = "acf_net-V2"


def build_dataloaders_v2(cfg: TrainConfigV2):
 print("[V2 Data] Loading manifest/splits...")
 print(f"[V2 Data] Manifest: {cfg.manifest_path}")
 print(f"[V2 Data] Splits: {cfg.split_path}")

 cases = load_or_create_manifest(cfg)
 splits = load_or_create_splits(cfg, cases)

 train_ds = BratsFastSurferDatasetV2(cfg=cfg, samples=splits["train"], training=True)
 val_ds = BratsFastSurferDatasetV2(cfg=cfg, samples=splits["val"], training=False)
 test_ds = BratsFastSurferDatasetV2(cfg=cfg, samples=splits["test"], training=False) if len(splits["test"]) else None

 print(f"[V2 Data] train cases: {len(train_ds)}")
 print(f"[V2 Data] val cases: {len(val_ds)}")
 print(f"[V2 Data] test cases: {len(test_ds) if test_ds is not None else 0}")
 print(f"[V2 Data] patch_size={cfg.patch_size}, batch_size={cfg.batch_size}")
 print(f"[V2 Data] tumor_patch_prob={cfg.tumor_patch_prob}, et_patch_prob={cfg.et_patch_prob}")
 print(f"[V2 Data] cache_dir={cfg.cache_dir}")

 loader_kwargs = dict(
 num_workers=cfg.num_workers,
 pin_memory=cfg.pin_memory,
 persistent_workers=cfg.persistent_workers if cfg.num_workers > 0 else False,
 )
 if cfg.num_workers > 0:
 loader_kwargs["prefetch_factor"] = cfg.prefetch_factor

 train_loader = DataLoader(
 train_ds,
 batch_size=cfg.batch_size,
 shuffle=True,
 **loader_kwargs,
 )

 val_loader = DataLoader(
 val_ds,
 batch_size=1,
 shuffle=False,
 **loader_kwargs,
 )

 test_loader = None
 if test_ds is not None:
 test_loader = DataLoader(
 test_ds,
 batch_size=1,
 shuffle=False,
 **loader_kwargs,
 )

 return train_loader, val_loader, test_loader, splits

In [ ]:
@torch.no_grad()
def validate_one_epoch_v2(
 model: nn.Module,
 loader: DataLoader,
 device: str,
 dice_loss: nn.Module,
 region_dice_loss: nn.Module,
 cfg: TrainConfigV2,
 epoch: Optional[int] = None,
 debug_shapes: bool = False,
) -> Dict[str, float]:
 model.eval()

 loss_meter = AverageMeter()
 class_dice_meter = AverageMeter()
 region_dice_meter = AverageMeter()

 region_meters = {
 "dice_wt": AverageMeter(),
 "dice_tc": AverageMeter(),
 "dice_et": AverageMeter(),
 }
 class_meters = {
 f"dice_class_{c}": AverageMeter()
 for c in range(1, cfg.num_classes)
 }

 desc = f"V2 Val epoch {epoch}" if epoch is not None else "V2 Validation"
 pbar = tqdm(loader, desc=desc, leave=False)

 for batch_idx, batch in enumerate(pbar):
 image = batch["image"].to(device, non_blocking=True)
 priors = batch["priors"].to(device, non_blocking=True)
 region = batch["region"].to(device, non_blocking=True)
 target = batch["seg"].to(device, non_blocking=True)

 with torch.cuda.amp.autocast(enabled=cfg.amp and device.startswith("cuda")):
 outputs = model(image, priors, region)
 loss, _ = compute_total_loss_v2(outputs, target, dice_loss, region_dice_loss, cfg)

 if debug_shapes and batch_idx == 0:
 print_debug_shapes_once(image, priors, region, outputs, prefix="[V2 DEBUG validation first batch]")

 class_metrics = dice_per_class_from_logits(outputs["logits_fused"], target, cfg.num_classes)
 region_metrics = brats_region_dice_from_logits(outputs["logits_fused"], target)

 bsz = image.size(0)
 loss_meter.update(float(loss.item()), bsz)
 class_dice_meter.update(class_metrics["mean_dice"], bsz)
 region_dice_meter.update(region_metrics["mean_region_dice"], bsz)

 for c in range(1, cfg.num_classes):
 class_meters[f"dice_class_{c}"].update(class_metrics[f"dice_class_{c}"], bsz)

 for key in ["dice_wt", "dice_tc", "dice_et"]:
 region_meters[key].update(region_metrics[key], bsz)

 pbar.set_postfix({
 "loss": f"{loss_meter.avg:.4f}",
 "region_dice": f"{region_dice_meter.avg:.4f}",
 })

 result = {
 "val_loss": loss_meter.avg,
 "val_mean_dice": class_dice_meter.avg,
 "val_mean_region_dice": region_dice_meter.avg,
 "val_dice_wt": region_meters["dice_wt"].avg,
 "val_dice_tc": region_meters["dice_tc"].avg,
 "val_dice_et": region_meters["dice_et"].avg,
 }

 for c in range(1, cfg.num_classes):
 result[f"val_dice_class_{c}"] = class_meters[f"dice_class_{c}"].avg

 return result


def train_one_epoch_v2(
 model: nn.Module,
 loader: DataLoader,
 optimizer: torch.optim.Optimizer,
 scaler: Optional[torch.cuda.amp.GradScaler],
 device: str,
 dice_loss: nn.Module,
 region_dice_loss: nn.Module,
 cfg: TrainConfigV2,
 epoch: Optional[int] = None,
 debug_shapes: bool = False,
) -> Dict[str, float]:
 model.train()

 loss_meter = AverageMeter()
 class_dice_meter = AverageMeter()
 region_dice_meter = AverageMeter()
 region_loss_meter = AverageMeter()

 desc = f"V2 Train epoch {epoch}" if epoch is not None else "V2 Training"
 pbar = tqdm(loader, desc=desc, leave=False)

 for batch_idx, batch in enumerate(pbar):
 image = batch["image"].to(device, non_blocking=True)
 priors = batch["priors"].to(device, non_blocking=True)
 region = batch["region"].to(device, non_blocking=True)
 target = batch["seg"].to(device, non_blocking=True)

 optimizer.zero_grad(set_to_none=True)

 with torch.cuda.amp.autocast(enabled=cfg.amp and device.startswith("cuda")):
 outputs = model(image, priors, region)
 loss, parts = compute_total_loss_v2(outputs, target, dice_loss, region_dice_loss, cfg)

 if debug_shapes and batch_idx == 0:
 print_debug_shapes_once(image, priors, region, outputs, prefix="[V2 DEBUG training first batch]")

 if scaler is not None and cfg.amp and device.startswith("cuda"):
 scaler.scale(loss).backward()
 scaler.unscale_(optimizer)
 torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip_norm)
 scaler.step(optimizer)
 scaler.update()
 else:
 loss.backward()
 torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip_norm)
 optimizer.step()

 if batch_idx == 0:
 print_gpu_memory("V2 after first training batch")

 with torch.no_grad():
 class_metrics = dice_per_class_from_logits(outputs["logits_fused"], target, cfg.num_classes)
 region_metrics = brats_region_dice_from_logits(outputs["logits_fused"], target)

 bsz = image.size(0)
 loss_meter.update(float(loss.item()), bsz)
 class_dice_meter.update(class_metrics["mean_dice"], bsz)
 region_dice_meter.update(region_metrics["mean_region_dice"], bsz)
 region_loss_meter.update(parts["loss_region"], bsz)

 pbar.set_postfix({
 "loss": f"{loss_meter.avg:.4f}",
 "reg_dice": f"{region_dice_meter.avg:.4f}",
 "reg_loss": f"{region_loss_meter.avg:.4f}",
 })

 return {
 "train_loss": loss_meter.avg,
 "train_mean_dice": class_dice_meter.avg,
 "train_mean_region_dice": region_dice_meter.avg,
 "train_region_loss": region_loss_meter.avg,
 }

In [ ]:
def fit_v2(cfg: TrainConfigV2) -> None:
 mount_drive_in_colab()
 prepare_v2_dirs(cfg)
 set_seed(cfg.seed)

 print("=" * 80)
 print("acf_net training startup")
 print("=" * 80)
 print(f"Device: {cfg.device}")
 if torch.cuda.is_available():
 print(f"GPU: {torch.cuda.get_device_name(0)}")
 print(f"Monitor metric: {cfg.monitor_metric}")
 print(f"Region loss enabled: {cfg.use_region_loss}, weight={cfg.region_loss_weight}")
 print(f"Tumor-aware sampling: {cfg.use_patch_sampling}")

 train_loader, val_loader, test_loader, splits = build_dataloaders_v2(cfg)

 print("\n[V2 Startup] Initializing model...")
 model = ACRSWAFNetV2(
 num_classes=cfg.num_classes,
 num_prior_channels=9,
 num_regions=cfg.num_regions,
 region_vocab=cfg.region_vocab,
 router_hidden=cfg.router_hidden,
 router_region_dim=cfg.router_region_dim,
 router_global_dim=cfg.router_global_dim,
 router_calibration_lambda=cfg.router_calibration_lambda,
 debug_shapes=cfg.DEBUG_SHAPES,
 return_voxel_weights=False,
 ).to(cfg.device)

 print_parameter_summary(
 model,
 model_name="acf_net",
 baseline_trainable_m=30.43,
 print_modules=True,
 )
 print_gpu_memory("V2 after model creation")

 optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

 scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
 optimizer,
 mode="max",
 factor=cfg.lr_plateau_factor,
 patience=cfg.lr_plateau_patience,
 min_lr=cfg.min_lr,
 )

 scaler = torch.cuda.amp.GradScaler(enabled=cfg.amp and cfg.device.startswith("cuda"))

 dice_loss = SoftDiceLoss(num_classes=cfg.num_classes)
 region_dice_loss = BraTSRegionDiceLoss(
 wt_weight=cfg.region_loss_wt_weight,
 tc_weight=cfg.region_loss_tc_weight,
 et_weight=cfg.region_loss_et_weight,
 )

 early_stopping = EarlyStopping(
 patience=cfg.patience,
 mode=cfg.monitor_mode,
 min_delta=cfg.min_delta,
 )

 initialize_zero_region_calibration(model, cfg)

 start_epoch = 0
 best_metric = -math.inf
 history: List[Dict] = []

 resume_path = cfg.resume_path
 if resume_path is None and cfg.AUTO_RESUME:
 resume_path = latest_checkpoint_path(cfg.checkpoint_dir)

 if resume_path is not None and os.path.exists(resume_path):
 try:
 print(f"[V2 Auto-resume] Attempting resume from: {resume_path}")
 ckpt = load_checkpoint(
 resume_path,
 model,
 optimizer,
 scaler,
 map_location=cfg.device,
 scheduler=scheduler,
 strict=True,
 )

 completed_epoch = int(ckpt["epoch"])
 start_epoch = completed_epoch + 1
 best_metric = float(ckpt.get("best_metric", best_metric))
 history = list(ckpt.get("history", []))

 if ckpt.get("early_stopping") is not None:
 early_stopping.load_state_dict(ckpt["early_stopping"])

 print("[V2 Auto-resume] Resume successful.")
 print(f" completed_epoch: {completed_epoch}")
 print(f" next_epoch: {start_epoch}")
 print(f" best_metric: {best_metric:.6f}")

 except RuntimeError as exc:
 print("[V2 Auto-resume] Incompatible checkpoint. Starting from scratch.")
 print(str(exc))
 if cfg.resume_path is not None:
 raise
 else:
 print("[V2 Auto-resume] No valid V2 checkpoint found. Starting from scratch.")

 history_path = os.path.join(cfg.log_dir, "history_v2.json")
 effective_max_epochs = cfg.smoke_test_epochs if cfg.smoke_test else cfg.max_epochs

 for epoch in range(start_epoch, effective_max_epochs):
 epoch_start = time.time()

 train_metrics = train_one_epoch_v2(
 model=model,
 loader=train_loader,
 optimizer=optimizer,
 scaler=scaler,
 device=cfg.device,
 dice_loss=dice_loss,
 region_dice_loss=region_dice_loss,
 cfg=cfg,
 epoch=epoch,
 debug_shapes=cfg.DEBUG_SHAPES and epoch == start_epoch,
 )

 if (epoch + 1) % cfg.val_every == 0:
 val_metrics = validate_one_epoch_v2(
 model=model,
 loader=val_loader,
 device=cfg.device,
 dice_loss=dice_loss,
 region_dice_loss=region_dice_loss,
 cfg=cfg,
 epoch=epoch,
 debug_shapes=False,
 )
 else:
 val_metrics = {
 "val_loss": float("nan"),
 "val_mean_dice": float("nan"),
 "val_mean_region_dice": float("nan"),
 "val_dice_wt": float("nan"),
 "val_dice_tc": float("nan"),
 "val_dice_et": float("nan"),
 }

 current_metric = val_metrics.get(cfg.monitor_metric, float("nan"))
 if math.isnan(current_metric):
 current_metric = best_metric

 scheduler.step(current_metric)

 epoch_time = time.time() - epoch_start

 row = {
 "epoch": epoch,
 "architecture_name": cfg.architecture_name,
 "lr": optimizer.param_groups[0]["lr"],
 "time_sec": epoch_time,
 **train_metrics,
 **val_metrics,
 }
 history.append(row)
 save_json({"history": history}, history_path)

 improved = current_metric > best_metric + cfg.min_delta
 if improved:
 best_metric = float(current_metric)

 stop = early_stopping.step(float(current_metric))

 if improved:
 best_path = os.path.join(cfg.checkpoint_dir, "best_model.pt")
 save_checkpoint(
 best_path,
 model,
 optimizer,
 scaler,
 epoch,
 best_metric,
 history,
 early_stopping,
 cfg,
 scheduler=scheduler,
 )
 print(f"[V2 Checkpoint] Saved best model: {best_path}")

 save_epoch_file = (
 cfg.SAVE_EVERY_EPOCH
 or (cfg.save_every > 0 and ((epoch + 1) % cfg.save_every == 0))
 )

 if save_epoch_file and not cfg.SAVE_BEST_ONLY:
 ckpt_path = os.path.join(cfg.checkpoint_dir, f"epoch_{epoch:04d}.pt")
 save_checkpoint(
 ckpt_path,
 model,
 optimizer,
 scaler,
 epoch,
 best_metric,
 history,
 early_stopping,
 cfg,
 scheduler=scheduler,
 )
 prune_old_epoch_checkpoints(cfg.checkpoint_dir, keep_last_k=cfg.keep_last_k)

 latest_path = os.path.join(cfg.checkpoint_dir, "latest.pt")
 save_checkpoint(
 latest_path,
 model,
 optimizer,
 scaler,
 epoch,
 best_metric,
 history,
 early_stopping,
 cfg,
 scheduler=scheduler,
 )

 print(
 f"V2 Epoch {epoch:03d} | "
 f"train_loss={row['train_loss']:.4f} | "
 f"train_region_dice={row['train_mean_region_dice']:.4f} | "
 f"val_region_dice={row['val_mean_region_dice']:.4f} | "
 f"val_wt={row.get('val_dice_wt', float('nan')):.4f} | "
 f"val_tc={row.get('val_dice_tc', float('nan')):.4f} | "
 f"val_et={row.get('val_dice_et', float('nan')):.4f} | "
 f"best={best_metric:.4f} | "
 f"lr={row['lr']:.2e} | "
 f"time={epoch_time:.1f}s"
 )

 if stop:
 print(f"[V2 Early stopping] Triggered at epoch {epoch}")
 break

 print("[V2] Training finished.")

run to train

In [ ]:
cfg_v2 = TrainConfigV2(
 brats_root="FILL IN WITH DIRECTORY FOR BRATS GLI TRAINING DATA",
 fastsurfer_root="FILL IN WITH DIRECTORY FOR FASTSURFER LABELED TRAINING OUTPUTS",

 # Isolated V2 outputs
 checkpoint_dir="FILL IN WITH DIRECTORY FOR MODEL CHECKPOINTS_v2",
 log_dir="FILL IN WITH DIRECTORY FOR TRAINING LOGS_v2",
 results_dir="FILL IN WITH DIRECTORY FOR EVALUATION RESULTS_v2",
 cache_dir="FILL IN WITH DIRECTORY FOR VALIDATION PRIOR CACHE_v2",

 # Keep split consistent with V1
 manifest_path="FILL IN WITH DIRECTORY FOR RUN MANIFEST/manifest.json",
 split_path="FILL IN WITH DIRECTORY FOR TRAIN VALIDATION TEST SPLITS/splits.json",

 # Colab runtime settings
 batch_size=1,
 num_workers=2,
 pin_memory=True,
 persistent_workers=True,
 prefetch_factor=2,
 amp=True,

 # V2 training settings
 max_epochs=120,
 lr=1e-4,
 patience=22,
 val_every=1,
 save_every=5,
 keep_last_k=4,

 # V2 loss/sampling
 use_patch_sampling=True,
 tumor_patch_prob=0.70,
 et_patch_prob=0.30,
 use_region_loss=True,
 region_loss_weight=0.35,
 region_loss_et_weight=1.35,

 AUTO_RESUME=True,
 DEBUG_SHAPES=True,
)

fit_v2(cfg_v2)

run for paths

In [ ]:
print("=" * 80)
print("V2 evaluation setup")
print("=" * 80)
print("Recommended final V2 reports:")
print("1. acf_net raw")
print("2. acf_net + postprocessing")
print("3. acf_net + TTA")
print("4. acf_net + TTA + postprocessing")
print()
print("Use validation only to tune postprocessing thresholds.")
print("Never tune thresholds on the test split.")
print()
print("Best V2 checkpoint path:")
print("FILL IN WITH DIRECTORY FOR MODEL CHECKPOINTS_v2/best_model.pt")
print()
print("V2 results directory:")
print("FILL IN WITH DIRECTORY FOR EVALUATION RESULTS_v2")

run for eval

In [ ]:
required_for_v2_eval = [
 "BratsFastSurferDatasetV2",
 "ACRSWAFNetV2",
 "load_or_build_priors",
 "remap_brats_labels",
 "load_volume",
 "zscore_nonzero",
]

missing_defs = [name for name in required_for_v2_eval if name not in globals()]

if missing_defs:
 print("Missing required definitions:")
 for name in missing_defs:
 print(" -", name)

 raise RuntimeError(
 "Run the V2 notebook definition cells above 'run for eval' first. "
 "Do not run fit_v2(cfg_v2). You only need the class/function definitions loaded."
 )

print("All required V2 eval definitions are loaded.")

In [ ]:
from pathlib import Path
from types import SimpleNamespace
import os
import copy
import json
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

try:
 from scipy.ndimage import binary_erosion, distance_transform_edt
 SCIPY_AVAILABLE = True
except Exception as exc:
 SCIPY_AVAILABLE = False
 print(f"[Warning] scipy unavailable for HD95: {repr(exc)}")

try:
 from sklearn.metrics import roc_auc_score, average_precision_score
 SKLEARN_AVAILABLE = True
except Exception as exc:
 SKLEARN_AVAILABLE = False
 print(f"[Warning] sklearn unavailable for AUROC/AUPRC: {repr(exc)}")


def v2_ensure_dir(path):
 os.makedirs(path, exist_ok=True)


BRATS_ROOT_V2 = (
 "FILL IN WITH DIRECTORY FOR GOOGLE DRIVE ROOTHasan Thesis/Datasets/BraTS-GLI/"
 "asnr-miccai-brats2023-gli-challenge-trainingdata/"
 "ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData"
)

# Match V1 final labeled evaluation behavior.
FASTSURFER_LABELED_ROOT_V2 = "FILL IN WITH DIRECTORY FOR FASTSURFER LABELED TRAINING OUTPUTS"

CHECKPOINT_DIR_V2 = "FILL IN WITH DIRECTORY FOR MODEL CHECKPOINTS_v2"
RESULTS_DIR_V2 = "FILL IN WITH DIRECTORY FOR EVALUATION RESULTS_v2"
CACHE_DIR_V2 = "FILL IN WITH DIRECTORY FOR VALIDATION PRIOR CACHE_v2_eval_outputGLItraining"

SPLIT_PATH_V2 = "FILL IN WITH DIRECTORY FOR TRAIN VALIDATION TEST SPLITS/splits.json"
MANIFEST_PATH_V2 = "FILL IN WITH DIRECTORY FOR RUN MANIFEST/manifest.json"

BEST_CHECKPOINT_V2 = os.path.join(CHECKPOINT_DIR_V2, "best_model.pt")


def build_eval_cfg_v2():
 if "cfg_v2" in globals():
 cfg = copy.deepcopy(cfg_v2)
 print("[Config] Reusing cfg_v2 as the base eval config.")
 else:
 cfg = SimpleNamespace()
 print("[Config] cfg_v2 not found. Using lightweight eval config.")

 cfg.brats_root = BRATS_ROOT_V2
 cfg.fastsurfer_root = FASTSURFER_LABELED_ROOT_V2
 cfg.fastsurfer_labeled_root = FASTSURFER_LABELED_ROOT_V2

 cfg.checkpoint_dir = CHECKPOINT_DIR_V2
 cfg.results_dir = RESULTS_DIR_V2
 cfg.cache_dir = CACHE_DIR_V2
 cfg.manifest_path = MANIFEST_PATH_V2
 cfg.split_path = SPLIT_PATH_V2
 cfg.best_checkpoint_path = BEST_CHECKPOINT_V2

 cfg.patch_size = getattr(cfg, "patch_size", (96, 96, 96))
 cfg.batch_size = 1
 cfg.use_patch_sampling = False

 # Use num_workers=0 for eval stability. This does not change metrics.
 cfg.num_workers = 0
 cfg.pin_memory = False
 cfg.persistent_workers = False
 cfg.prefetch_factor = 2

 cfg.cache_priors = True
 cfg.debug_resampling_shapes = False
 cfg.DEBUG_SHAPES = False

 cfg.num_classes = getattr(cfg, "num_classes", 4)
 cfg.num_regions = getattr(cfg, "num_regions", 6)
 cfg.region_vocab = getattr(cfg, "region_vocab", 16)
 cfg.router_hidden = getattr(cfg, "router_hidden", 32)
 cfg.router_region_dim = getattr(cfg, "router_region_dim", 8)
 cfg.router_global_dim = getattr(cfg, "router_global_dim", 16)
 cfg.router_calibration_lambda = getattr(cfg, "router_calibration_lambda", 1.0)

 cfg.device = "cuda" if torch.cuda.is_available() else "cpu"

 return cfg


cfg_v2_eval = build_eval_cfg_v2()

v2_ensure_dir(cfg_v2_eval.results_dir)
v2_ensure_dir(cfg_v2_eval.cache_dir)

print("=" * 80)
print("ACF-Net V2 evaluation config matching V1 labeled-eval behavior")
print("=" * 80)
print(f"BraTS root: {cfg_v2_eval.brats_root}")
print(f"FastSurfer labeled root: {cfg_v2_eval.fastsurfer_labeled_root}")
print(f"Best V2 checkpoint: {cfg_v2_eval.best_checkpoint_path}")
print(f"Split file: {cfg_v2_eval.split_path}")
print(f"Results dir: {cfg_v2_eval.results_dir}")
print(f"Prior cache dir: {cfg_v2_eval.cache_dir}")
print(f"Device: {cfg_v2_eval.device}")

assert os.path.exists(cfg_v2_eval.best_checkpoint_path), (
 f"Missing V2 best checkpoint: {cfg_v2_eval.best_checkpoint_path}"
)
assert os.path.exists(cfg_v2_eval.split_path), (
 f"Missing split file: {cfg_v2_eval.split_path}"
)
assert os.path.exists(cfg_v2_eval.fastsurfer_labeled_root), (
 f"Missing FastSurfer labeled root: {cfg_v2_eval.fastsurfer_labeled_root}"
)

In [ ]:
def v2_case_id_from_sample(sample):
 if isinstance(sample, str):
 return sample

 for key in ["case_id", "id", "name"]:
 if key in sample:
 return str(sample[key])

 if "mri" in sample and isinstance(sample["mri"], dict):
 first_path = next(iter(sample["mri"].values()))
 return Path(first_path).parent.name

 raise KeyError(f"Could not infer case id from sample keys: {list(sample.keys())}")


def v2_load_eval_splits(cfg):
 with open(cfg.split_path, "r") as f:
 splits = json.load(f)

 train_ids = [v2_case_id_from_sample(s) for s in splits["train"]]
 val_ids = [v2_case_id_from_sample(s) for s in splits["val"]]
 test_ids = [v2_case_id_from_sample(s) for s in splits["test"]]

 print("=" * 80)
 print("Loaded split ids")
 print("=" * 80)
 print(f"Train cases in split: {len(train_ids)}")
 print(f"Val cases in split: {len(val_ids)}")
 print(f"Test cases in split: {len(test_ids)}")

 return train_ids, val_ids, test_ids


def v2_expected_mri_paths(case_id, brats_root):
 case_dir = Path(brats_root) / case_id
 return {
 "t1": str(case_dir / f"{case_id}-t1n.nii.gz"),
 "t1ce": str(case_dir / f"{case_id}-t1c.nii.gz"),
 "t2": str(case_dir / f"{case_id}-t2w.nii.gz"),
 "flair": str(case_dir / f"{case_id}-t2f.nii.gz"),
 }


def v2_expected_seg_path(case_id, brats_root):
 return str(Path(brats_root) / case_id / f"{case_id}-seg.nii.gz")


def v2_resolve_brats_root(cfg, val_ids, test_ids):
 probe_ids = list(val_ids[:3]) + list(test_ids[:3])

 def root_has_probe(root):
 for case_id in probe_ids:
 paths = v2_expected_mri_paths(case_id, root)
 seg_path = v2_expected_seg_path(case_id, root)
 if os.path.exists(paths["t1"]) and os.path.exists(seg_path):
 return True
 return False

 if os.path.exists(cfg.brats_root) and root_has_probe(cfg.brats_root):
 print(f"[BraTS root] Using configured root: {cfg.brats_root}")
 return cfg.brats_root

 print("[Warning] Configured BraTS root did not contain the probe cases.")
 print("[Search] Looking for the first split case under FILL IN WITH DIRECTORY FOR GOOGLE DRIVE ROOTHasan Thesis/Datasets ...")

 search_root = Path("FILL IN WITH DIRECTORY FOR GOOGLE DRIVE ROOTHasan Thesis/Datasets")
 first_case = probe_ids[0]

 matches = list(search_root.rglob(first_case))
 for match in matches:
 if match.is_dir():
 candidate_root = str(match.parent)
 paths = v2_expected_mri_paths(first_case, candidate_root)
 seg_path = v2_expected_seg_path(first_case, candidate_root)
 if os.path.exists(paths["t1"]) and os.path.exists(seg_path):
 print(f"[BraTS root] Auto-detected root: {candidate_root}")
 return candidate_root

 raise RuntimeError(
 "Could not find a BraTS root containing the split cases. "
 "Check BRATS_ROOT_V2 manually."
 )


def v2_build_labeled_sample(case_id, brats_root, fastsurfer_root):
 case_dir = Path(brats_root) / case_id
 fs_dir = Path(fastsurfer_root) / case_id / "mri"

 return {
 "id": case_id,
 "case_id": case_id,
 "mri": {
 "t1": str(case_dir / f"{case_id}-t1n.nii.gz"),
 "t1ce": str(case_dir / f"{case_id}-t1c.nii.gz"),
 "t2": str(case_dir / f"{case_id}-t2w.nii.gz"),
 "flair": str(case_dir / f"{case_id}-t2f.nii.gz"),
 },
 "seg": str(case_dir / f"{case_id}-seg.nii.gz"),
 "fastsurfer": {
 # These exact keys are required by derive_anatomy_priors_in_fastsurfer_space.
 "dkt": str(fs_dir / "aparc.DKTatlas+aseg.deep.mgz"),
 "aseg": str(fs_dir / "aseg.auto_noCCseg.mgz"),
 "mask": str(fs_dir / "mask.mgz"),
 },
 }


def v2_validate_sample_files(sample):
 missing = []

 for _, path in sample["mri"].items():
 if not os.path.exists(path):
 missing.append(path)

 if not os.path.exists(sample["seg"]):
 missing.append(sample["seg"])

 for _, path in sample["fastsurfer"].items():
 if not os.path.exists(path):
 missing.append(path)

 return missing


def v2_make_eval_samples(case_ids, cfg, split_name):
 samples = []
 missing_rows = []

 for case_id in case_ids:
 sample = v2_build_labeled_sample(
 case_id=case_id,
 brats_root=cfg.brats_root,
 fastsurfer_root=cfg.fastsurfer_labeled_root,
 )

 missing = v2_validate_sample_files(sample)

 if missing:
 missing_rows.append({
 "split": split_name,
 "case_id": case_id,
 "missing_count": len(missing),
 "first_missing": missing[0],
 })
 continue

 samples.append(sample)

 missing_df = pd.DataFrame(missing_rows)

 print("=" * 80)
 print(f"{split_name} labeled sample check using outputGLItraining")
 print("=" * 80)
 print(f"Cases in split: {len(case_ids)}")
 print(f"Cases with usable inputs: {len(samples)}")
 print(f"Cases missing files: {len(missing_df)}")

 if len(missing_df):
 display(missing_df.head(20))

 return samples, missing_df


train_ids_v2, val_ids_v2, test_ids_v2 = v2_load_eval_splits(cfg_v2_eval)
cfg_v2_eval.brats_root = v2_resolve_brats_root(cfg_v2_eval, val_ids_v2, test_ids_v2)

In [ ]:
V2_BRATS_REGIONS = {
 "wt": (1, 2, 3),
 "tc": (1, 3),
 "et": (3,),
}


def v2_safe_float(x):
 try:
 if x is None:
 return float("nan")
 return float(x)
 except Exception:
 return float("nan")


def v2_nanmean(values):
 arr = np.asarray(values, dtype=np.float64)
 if arr.size == 0 or np.all(np.isnan(arr)):
 return float("nan")
 return float(np.nanmean(arr))


def v2_mask_from_labels(arr, labels):
 mask = np.zeros_like(arr, dtype=bool)
 for label in labels:
 mask |= arr == label
 return mask


def v2_dice_binary(pred_mask, target_mask, eps=1e-6):
 pred_mask = pred_mask.astype(bool)
 target_mask = target_mask.astype(bool)

 pred_sum = int(pred_mask.sum())
 target_sum = int(target_mask.sum())

 if pred_sum == 0 and target_sum == 0:
 return 1.0
 if pred_sum == 0 or target_sum == 0:
 return 0.0

 inter = int(np.logical_and(pred_mask, target_mask).sum())
 return float((2.0 * inter + eps) / (pred_sum + target_sum + eps))


def v2_surface(mask):
 mask = mask.astype(bool)
 if mask.sum() == 0:
 return mask

 eroded = binary_erosion(mask, structure=np.ones((3, 3, 3)), border_value=0)
 return np.logical_xor(mask, eroded)


def v2_hd95_binary(pred_mask, target_mask):
 # Match V1 output note: HD95 in voxel units.
 spacing = (1.0, 1.0, 1.0)

 if not SCIPY_AVAILABLE:
 return float("nan")

 pred_mask = pred_mask.astype(bool)
 target_mask = target_mask.astype(bool)

 pred_sum = int(pred_mask.sum())
 target_sum = int(target_mask.sum())

 if pred_sum == 0 and target_sum == 0:
 return 0.0
 if pred_sum == 0 or target_sum == 0:
 return float("nan")

 pred_surface = v2_surface(pred_mask)
 target_surface = v2_surface(target_mask)

 if pred_surface.sum() == 0 or target_surface.sum() == 0:
 return float("nan")

 dt_pred = distance_transform_edt(~pred_surface, sampling=spacing)
 dt_target = distance_transform_edt(~target_surface, sampling=spacing)

 d_pred_to_target = dt_target[pred_surface]
 d_target_to_pred = dt_pred[target_surface]
 distances = np.concatenate([d_pred_to_target, d_target_to_pred])

 if distances.size == 0:
 return float("nan")

 return float(np.percentile(distances, 95))


def v2_brats_region_metrics(pred, target):
 out = {}
 dice_vals = []
 hd95_vals = []

 for name, labels in V2_BRATS_REGIONS.items():
 pred_mask = v2_mask_from_labels(pred, labels)
 target_mask = v2_mask_from_labels(target, labels)

 dice = v2_dice_binary(pred_mask, target_mask)
 hd95 = v2_hd95_binary(pred_mask, target_mask)

 out[f"dice_{name}"] = dice
 out[f"hd95_{name}"] = hd95

 dice_vals.append(dice)
 hd95_vals.append(hd95)

 out["mean_region_dice"] = float(np.mean(dice_vals))
 out["mean_hd95"] = v2_nanmean(hd95_vals)

 return out


def v2_wt_detection_metrics(pred, target, probs_np):
 pred_wt = pred > 0
 target_wt = target > 0
 p_wt = 1.0 - probs_np[0]

 tp = int(np.logical_and(pred_wt, target_wt).sum())
 fp = int(np.logical_and(pred_wt, ~target_wt).sum())
 fn = int(np.logical_and(~pred_wt, target_wt).sum())

 pred_sum = int(pred_wt.sum())
 target_sum = int(target_wt.sum())

 if pred_sum == 0 and target_sum == 0:
 precision = 1.0
 recall = 1.0
 elif pred_sum == 0 and target_sum > 0:
 precision = float("nan")
 recall = 0.0
 elif pred_sum > 0 and target_sum == 0:
 precision = 0.0
 recall = float("nan")
 else:
 precision = tp / max(tp + fp, 1)
 recall = tp / max(tp + fn, 1)

 auroc = float("nan")
 auprc = float("nan")
 au_status = "not_computed"

 if SKLEARN_AVAILABLE:
 y_true = target_wt.astype(np.uint8).reshape(-1)
 y_score = p_wt.astype(np.float32).reshape(-1)

 if np.unique(y_true).size >= 2:
 try:
 auroc = float(roc_auc_score(y_true, y_score))
 auprc = float(average_precision_score(y_true, y_score))
 au_status = "ok"
 except Exception as exc:
 au_status = f"metric_error:{repr(exc)}"
 else:
 au_status = "single_class_target"

 return {
 "auroc_wt": v2_safe_float(auroc),
 "auprc_wt": v2_safe_float(auprc),
 "precision_wt": v2_safe_float(precision),
 "recall_wt": v2_safe_float(recall),
 "tp_wt": tp,
 "fp_wt": fp,
 "fn_wt": fn,
 "pred_wt_voxels": pred_sum,
 "target_wt_voxels": target_sum,
 "auroc_auprc_status": au_status,
 }


def v2_extract_case_id(batch):
 case_id = batch.get("id", batch.get("case_id", "unknown"))
 if isinstance(case_id, (list, tuple)):
 return str(case_id[0])
 return str(case_id)


print("[V2 metrics] WT={1,2,3}, TC={1,3}, ET={3}")
print("[V2 metrics] HD95 uses voxel units to match V1 labeled-eval note.")

In [ ]:
def build_v2_eval_loaders_matching_v1(cfg, val_ids, test_ids):
 print("=" * 80)
 print("Building V2 evaluation loaders matching V1 labeled-eval behavior")
 print("=" * 80)

 val_samples, missing_val_df = v2_make_eval_samples(
 case_ids=val_ids,
 cfg=cfg,
 split_name="val",
 )

 test_samples, missing_test_df = v2_make_eval_samples(
 case_ids=test_ids,
 cfg=cfg,
 split_name="test",
 )

 if len(val_samples) == 0:
 raise RuntimeError("No usable validation samples found.")

 if len(test_samples) == 0:
 raise RuntimeError("No usable test samples found.")

 val_cfg = copy.deepcopy(cfg)
 val_cfg.use_patch_sampling = False
 val_cfg.fastsurfer_root = cfg.fastsurfer_labeled_root

 test_cfg = copy.deepcopy(cfg)
 test_cfg.use_patch_sampling = False
 test_cfg.fastsurfer_root = cfg.fastsurfer_labeled_root

 val_ds = BratsFastSurferDatasetV2(
 cfg=val_cfg,
 samples=val_samples,
 training=False,
 )

 test_ds = BratsFastSurferDatasetV2(
 cfg=test_cfg,
 samples=test_samples,
 training=False,
 )

 loader_kwargs = dict(
 num_workers=cfg.num_workers,
 pin_memory=cfg.pin_memory,
 persistent_workers=False,
 )

 val_loader = DataLoader(
 val_ds,
 batch_size=1,
 shuffle=False,
 **loader_kwargs,
 )

 test_loader = DataLoader(
 test_ds,
 batch_size=1,
 shuffle=False,
 **loader_kwargs,
 )

 print("=" * 80)
 print("Final V2 eval loader sizes")
 print("=" * 80)
 print(f"Validation loader cases: {len(val_ds)}")
 print(f"Test loader cases: {len(test_ds)}")
 print(f"FastSurfer root used: {cfg.fastsurfer_labeled_root}")

 return val_loader, test_loader, missing_val_df, missing_test_df


v2_val_loader, v2_test_loader, v2_missing_val_df, v2_missing_test_df = (
 build_v2_eval_loaders_matching_v1(cfg_v2_eval, val_ids_v2, test_ids_v2)
)

In [ ]:
def v2_safe_torch_load(path, map_location="cpu"):
 try:
 return torch.load(path, map_location=map_location, weights_only=False)
 except TypeError:
 return torch.load(path, map_location=map_location)


def build_v2_eval_model(cfg):
 model = ACRSWAFNetV2(
 num_classes=cfg.num_classes,
 num_prior_channels=9,
 num_regions=cfg.num_regions,
 region_vocab=cfg.region_vocab,
 router_hidden=cfg.router_hidden,
 router_region_dim=cfg.router_region_dim,
 router_global_dim=cfg.router_global_dim,
 router_calibration_lambda=cfg.router_calibration_lambda,
 debug_shapes=False,
 return_voxel_weights=False,
 ).to(cfg.device)

 return model


def load_v2_best_model(cfg):
 print("=" * 80)
 print("Loading ACF-Net V2 best checkpoint")
 print("=" * 80)
 print(f"Checkpoint path: {cfg.best_checkpoint_path}")

 model = build_v2_eval_model(cfg)
 ckpt = v2_safe_torch_load(cfg.best_checkpoint_path, map_location=cfg.device)

 if isinstance(ckpt, dict) and "model_state" in ckpt:
 state_dict = ckpt["model_state"]
 elif isinstance(ckpt, dict) and "state_dict" in ckpt:
 state_dict = ckpt["state_dict"]
 elif isinstance(ckpt, dict):
 state_dict = ckpt
 else:
 raise RuntimeError("Unsupported checkpoint format.")

 model.load_state_dict(state_dict, strict=True)
 model.eval()

 best_epoch = int(ckpt.get("epoch", -1)) if isinstance(ckpt, dict) else -1
 best_metric = v2_safe_float(ckpt.get("best_metric", float("nan"))) if isinstance(ckpt, dict) else float("nan")

 print("[V2 checkpoint] Load successful.")
 print(f"[V2 checkpoint] best_epoch: {best_epoch}")
 print(f"[V2 checkpoint] best_metric: {best_metric:.6f}")

 return model, ckpt, best_epoch, best_metric


v2_best_model, v2_best_ckpt, v2_best_epoch, v2_best_metric = load_v2_best_model(cfg_v2_eval)

In [ ]:
@torch.no_grad()
def evaluate_v2_loader(model, loader, cfg, split_name):
 model.eval()
 rows = []

 print("=" * 80)
 print(f"Evaluating V2 checkpoint on {split_name}")
 print("=" * 80)

 if split_name == "test":
 print("[Test note] Test metrics are computed on held-out splits['test'].")
 print("[Anatomy note] Labeled test metrics use outputGLItraining FastSurfer priors.")
 print("[Hausdorff note] HD95 is computed in voxel units.")

 for batch in tqdm(loader, desc=f"V2 {split_name}", leave=True):
 case_id = v2_extract_case_id(batch)

 image = batch["image"].to(cfg.device, non_blocking=True)
 priors = batch["priors"].to(cfg.device, non_blocking=True)
 region = batch["region"].to(cfg.device, non_blocking=True)
 target = batch["seg"].to(cfg.device, non_blocking=True)

 outputs = model(image, priors, region)

 if isinstance(outputs, dict):
 if "logits_fused" in outputs:
 logits = outputs["logits_fused"]
 elif "logits" in outputs:
 logits = outputs["logits"]
 else:
 raise KeyError(f"Could not find logits in model output keys: {list(outputs.keys())}")
 else:
 logits = outputs

 probs = torch.softmax(logits, dim=1)
 pred = torch.argmax(logits, dim=1)

 pred_np = pred[0].detach().cpu().numpy().astype(np.int16)
 target_np = target[0].detach().cpu().numpy().astype(np.int16)
 probs_np = probs[0].detach().cpu().numpy().astype(np.float32)

 row = {
 "split": split_name,
 "case_id": case_id,
 "model_variant": "ACF-Net V2",
 }

 row.update(v2_brats_region_metrics(pred_np, target_np))
 row.update(v2_wt_detection_metrics(pred_np, target_np, probs_np))

 rows.append(row)

 del outputs, logits, probs, pred
 if torch.cuda.is_available():
 torch.cuda.empty_cache()

 df = pd.DataFrame(rows)

 print(f"[V2 {split_name}] cases evaluated: {len(df)}")

 if len(df):
 print(f"[V2 {split_name}] auroc_wt: {v2_nanmean(df['auroc_wt'].values):.6f}")
 print(f"[V2 {split_name}] auprc_wt: {v2_nanmean(df['auprc_wt'].values):.6f}")
 print(f"[V2 {split_name}] dice_wt: {v2_nanmean(df['dice_wt'].values):.6f}")
 print(f"[V2 {split_name}] dice_tc: {v2_nanmean(df['dice_tc'].values):.6f}")
 print(f"[V2 {split_name}] dice_et: {v2_nanmean(df['dice_et'].values):.6f}")
 print(f"[V2 {split_name}] mean_region_dice: {v2_nanmean(df['mean_region_dice'].values):.6f}")
 print(f"[V2 {split_name}] precision_wt: {v2_nanmean(df['precision_wt'].values):.6f}")
 print(f"[V2 {split_name}] recall_wt: {v2_nanmean(df['recall_wt'].values):.6f}")
 print(f"[V2 {split_name}] hd95_wt: {v2_nanmean(df['hd95_wt'].values):.6f}")
 print(f"[V2 {split_name}] hd95_tc: {v2_nanmean(df['hd95_tc'].values):.6f}")
 print(f"[V2 {split_name}] hd95_et: {v2_nanmean(df['hd95_et'].values):.6f}")
 print(f"[V2 {split_name}] mean_hd95: {v2_nanmean(df['mean_hd95'].values):.6f}")

 return df


def summarize_v2_split(df, split_name, best_epoch, best_metric):
 prefix = f"{split_name}_"

 summary = {
 "model_variant": "ACF-Net V2",
 "best_epoch": int(best_epoch),
 "mean_val_dice": v2_safe_float(best_metric),
 "checkpoint_best_metric": v2_safe_float(best_metric),
 f"{prefix}num_cases": int(len(df)),
 }

 metric_cols = [
 "auroc_wt",
 "auprc_wt",
 "dice_wt",
 "dice_tc",
 "dice_et",
 "mean_region_dice",
 "precision_wt",
 "recall_wt",
 "hd95_wt",
 "hd95_tc",
 "hd95_et",
 "mean_hd95",
 ]

 for col in metric_cols:
 summary[f"{prefix}{col}"] = (
 v2_nanmean(df[col].values) if col in df.columns else float("nan")
 )

 return summary


v2_val_results_df = evaluate_v2_loader(
 model=v2_best_model,
 loader=v2_val_loader,
 cfg=cfg_v2_eval,
 split_name="val",
)

v2_test_results_df = evaluate_v2_loader(
 model=v2_best_model,
 loader=v2_test_loader,
 cfg=cfg_v2_eval,
 split_name="test",
)

v2_val_summary = summarize_v2_split(
 df=v2_val_results_df,
 split_name="val",
 best_epoch=v2_best_epoch,
 best_metric=v2_best_metric,
)

v2_test_summary = summarize_v2_split(
 df=v2_test_results_df,
 split_name="test",
 best_epoch=v2_best_epoch,
 best_metric=v2_best_metric,
)

v2_summary_row = {
 **v2_val_summary,
 **{
 k: v
 for k, v in v2_test_summary.items()
 if k not in {
 "model_variant",
 "best_epoch",
 "mean_val_dice",
 "checkpoint_best_metric",
 }
 },
}

v2_summary_row["validation_definition"] = (
 "best validation Dice stored in best_model.pt and sanity-checked on splits['val']"
)
v2_summary_row["test_definition"] = (
 "held-out splits['test'] evaluated once after checkpoint selection"
)
v2_summary_row["test_fastsurfer_root_used"] = cfg_v2_eval.fastsurfer_labeled_root
v2_summary_row["test_cases_evaluated"] = int(len(v2_test_results_df))
v2_summary_row["test_cases_with_labels"] = int(len(v2_test_results_df))
v2_summary_row["test_cases_missing_fastsurfer"] = int(len(v2_missing_test_df))
v2_summary_row["best_checkpoint_path"] = cfg_v2_eval.best_checkpoint_path
v2_summary_row["split_path"] = cfg_v2_eval.split_path
v2_summary_row["test_cache_dir"] = cfg_v2_eval.cache_dir

v2_summary_results_df = pd.DataFrame([v2_summary_row])

requested_v2_test_cols = [
 "test_auroc_wt",
 "test_auprc_wt",
 "test_dice_wt",
 "test_dice_tc",
 "test_dice_et",
 "test_mean_region_dice",
 "test_precision_wt",
 "test_recall_wt",
 "test_hd95_wt",
 "test_hd95_tc",
 "test_hd95_et",
 "test_mean_hd95",
]

v2_requested_test_metrics_df = v2_summary_results_df[requested_v2_test_cols].copy()

summary_path = os.path.join(cfg_v2_eval.results_dir, "acf_net_v2_eval_summary_v1_comparable.csv")
test_metrics_path = os.path.join(cfg_v2_eval.results_dir, "acf_net_v2_requested_test_metrics_v1_comparable.csv")
per_case_val_path = os.path.join(cfg_v2_eval.results_dir, "acf_net_v2_val_per_case_metrics_v1_comparable.csv")
per_case_test_path = os.path.join(cfg_v2_eval.results_dir, "acf_net_v2_test_per_case_metrics_v1_comparable.csv")
missing_val_path = os.path.join(cfg_v2_eval.results_dir, "acf_net_v2_missing_val_samples_v1_comparable.csv")
missing_test_path = os.path.join(cfg_v2_eval.results_dir, "acf_net_v2_missing_test_samples_v1_comparable.csv")

v2_summary_results_df.to_csv(summary_path, index=False)
v2_requested_test_metrics_df.to_csv(test_metrics_path, index=False)
v2_val_results_df.to_csv(per_case_val_path, index=False)
v2_test_results_df.to_csv(per_case_test_path, index=False)
v2_missing_val_df.to_csv(missing_val_path, index=False)
v2_missing_test_df.to_csv(missing_test_path, index=False)

print("=" * 80)
print("configured V2 test metrics, V1-comparable")
print("=" * 80)
display(v2_requested_test_metrics_df)

print("=" * 80)
print("Full V2 summary, V1-comparable")
print("=" * 80)
display(v2_summary_results_df)

print("=" * 80)
print("Saved files")
print("=" * 80)
print(f"Full summary: {summary_path}")
print(f"configured test table: {test_metrics_path}")
print(f"Val per-case metrics: {per_case_val_path}")
print(f"Test per-case metrics: {per_case_test_path}")
print(f"Missing val samples: {missing_val_path}")
print(f"Missing test samples: {missing_test_path}")

In [ ]:
# Expert weight analysis setup
import os
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

EXPERT_NAMES = ["context_expert", "boundary_expert", "anatomy_expert"]
REGION_NAMES = {
 0: "background_or_other",
 1: "left_white_matter",
 2: "right_white_matter",
 3: "cortex",
 4: "periventricular",
 5: "deep_gray",
}


def resolve_expert_checkpoint_path(cfg=None):
 candidates = []

 if cfg is not None and hasattr(cfg, "best_checkpoint_path"):
 candidates.append(str(cfg.best_checkpoint_path))

 checkpoint_dirs = []
 if cfg is not None and hasattr(cfg, "checkpoint_dir"):
 checkpoint_dirs.append(str(cfg.checkpoint_dir))
 if "CHECKPOINT_DIR_V2" in globals():
 checkpoint_dirs.append(str(CHECKPOINT_DIR_V2))

 for directory in checkpoint_dirs:
 candidates.extend([
 os.path.join(directory, "best.pt"),
 os.path.join(directory, "best_model.pt"),
 ])

 candidates.extend([
 "best.pt",
 "best_model.pt",
 "/content/best.pt",
 "/content/best_model.pt",
 ])

 seen = set()
 unique_candidates = []
 for path in candidates:
 if path not in seen:
 unique_candidates.append(path)
 seen.add(path)

 for path in unique_candidates:
 if os.path.exists(path):
 return path, unique_candidates

 raise FileNotFoundError(
 "No best checkpoint found. Checked:\n" + "\n".join(unique_candidates)
 )


EXPERT_ANALYSIS_DIR = os.path.join(
 getattr(cfg_v2_eval, "results_dir", "/content"),
 "expert_weight_analysis",
)
os.makedirs(EXPERT_ANALYSIS_DIR, exist_ok=True)

EXPERT_CHECKPOINT_PATH, EXPERT_CHECKPOINT_CANDIDATES = resolve_expert_checkpoint_path(cfg_v2_eval)

print("=" * 80)
print("Expert weight analysis setup")
print("=" * 80)
print(f"Checkpoint path: {EXPERT_CHECKPOINT_PATH}")
print(f"Output directory: {EXPERT_ANALYSIS_DIR}")
print(f"Experts: {EXPERT_NAMES}")
print("Region labels:")
for region_id, region_name in REGION_NAMES.items():
 print(f" {region_id}: {region_name}")

In [ ]:
# Checkpoint-level router tensor audit

def expert_safe_torch_load(path, map_location="cpu"):
 if "v2_safe_torch_load" in globals():
 return v2_safe_torch_load(path, map_location=map_location)
 try:
 return torch.load(path, map_location=map_location, weights_only=False)
 except TypeError:
 return torch.load(path, map_location=map_location)


def extract_state_dict_from_checkpoint(ckpt):
 if isinstance(ckpt, dict) and "model_state" in ckpt:
 return ckpt["model_state"]
 if isinstance(ckpt, dict) and "state_dict" in ckpt:
 return ckpt["state_dict"]
 if isinstance(ckpt, dict):
 tensor_values = [v for v in ckpt.values() if torch.is_tensor(v)]
 if len(tensor_values):
 return ckpt
 raise RuntimeError("Unsupported checkpoint format for tensor extraction.")


def tensor_summary_row(name, tensor):
 arr = tensor.detach().float().cpu().reshape(-1).numpy()
 row = {
 "tensor": name,
 "shape": "x".join(str(x) for x in tensor.shape),
 "num_values": int(arr.size),
 "mean": float(np.mean(arr)) if arr.size else float("nan"),
 "std": float(np.std(arr)) if arr.size else float("nan"),
 "min": float(np.min(arr)) if arr.size else float("nan"),
 "q25": float(np.quantile(arr, 0.25)) if arr.size else float("nan"),
 "median": float(np.quantile(arr, 0.50)) if arr.size else float("nan"),
 "q75": float(np.quantile(arr, 0.75)) if arr.size else float("nan"),
 "max": float(np.max(arr)) if arr.size else float("nan"),
 "mean_abs": float(np.mean(np.abs(arr))) if arr.size else float("nan"),
 "l2_norm": float(np.linalg.norm(arr)) if arr.size else float("nan"),
 "nonzero_values": int(np.count_nonzero(arr)) if arr.size else 0,
 }
 return row


expert_ckpt = expert_safe_torch_load(EXPERT_CHECKPOINT_PATH, map_location="cpu")
expert_state_dict = extract_state_dict_from_checkpoint(expert_ckpt)

router_tensor_names = [
 name for name in expert_state_dict.keys()
 if name.startswith("router.") or ".router." in name
]

expert_head_tensor_names = [
 name for name in expert_state_dict.keys()
 if name.startswith(("context_expert.", "boundary_expert.", "anatomy_expert."))
 or any(f".{expert_name}." in name for expert_name in EXPERT_NAMES)
]

expert_tensor_rows = []
for name in router_tensor_names + expert_head_tensor_names:
 value = expert_state_dict[name]
 if torch.is_tensor(value):
 expert_tensor_rows.append(tensor_summary_row(name, value))

expert_tensor_summary_df = pd.DataFrame(expert_tensor_rows)
expert_tensor_summary_path = os.path.join(EXPERT_ANALYSIS_DIR, "checkpoint_router_and_expert_tensor_summary.csv")
expert_tensor_summary_df.to_csv(expert_tensor_summary_path, index=False)

checkpoint_meta = {
 "checkpoint_path": EXPERT_CHECKPOINT_PATH,
 "epoch": int(expert_ckpt.get("epoch", -1)) if isinstance(expert_ckpt, dict) else None,
 "best_metric": float(expert_ckpt.get("best_metric", float("nan"))) if isinstance(expert_ckpt, dict) else None,
 "num_router_tensors": len(router_tensor_names),
 "num_expert_head_tensors": len(expert_head_tensor_names),
}
checkpoint_meta_path = os.path.join(EXPERT_ANALYSIS_DIR, "checkpoint_metadata.json")
with open(checkpoint_meta_path, "w") as f:
 json.dump(checkpoint_meta, f, indent=2)

print("=" * 80)
print("Checkpoint router and expert tensor audit")
print("=" * 80)
print(json.dumps(checkpoint_meta, indent=2))
print(f"Saved tensor summary: {expert_tensor_summary_path}")

if len(expert_tensor_summary_df):
 display_cols = ["tensor", "shape", "mean", "std", "min", "median", "max", "mean_abs", "l2_norm"]
 display(expert_tensor_summary_df[display_cols])
else:
 print("No router or expert-head tensors matched the expected names.")

In [ ]:
# Static router values from the best checkpoint

def get_state_tensor(state_dict, suffix):
 matches = [name for name in state_dict if name.endswith(suffix) and torch.is_tensor(state_dict[name])]
 if not matches:
 return None, []
 preferred = [name for name in matches if name.startswith("router.")]
 selected = preferred[0] if preferred else matches[0]
 return state_dict[selected].detach().float().cpu(), matches


calibration_tensor, calibration_matches = get_state_tensor(expert_state_dict, "router.calibration_scores")
log_tau_tensor, log_tau_matches = get_state_tensor(expert_state_dict, "router.log_tau")
score_layer_w, score_layer_w_matches = get_state_tensor(expert_state_dict, "router.score_mlp.2.weight")
score_layer_b, score_layer_b_matches = get_state_tensor(expert_state_dict, "router.score_mlp.2.bias")
first_layer_w, first_layer_w_matches = get_state_tensor(expert_state_dict, "router.score_mlp.0.weight")
first_layer_b, first_layer_b_matches = get_state_tensor(expert_state_dict, "router.score_mlp.0.bias")
region_embed, region_embed_matches = get_state_tensor(expert_state_dict, "router.region_embed.weight")

if calibration_tensor is None:
 raise KeyError("router.calibration_scores was not found in the checkpoint state dict.")

calibration_arr = calibration_tensor.numpy()
if calibration_arr.shape[0] == len(EXPERT_NAMES):
 calibration_rows = []
 for expert_idx, expert_name in enumerate(EXPERT_NAMES):
 for region_idx in range(calibration_arr.shape[1]):
 calibration_rows.append({
 "expert_id": expert_idx + 1,
 "expert_name": expert_name,
 "region_id": region_idx,
 "region_name": REGION_NAMES.get(region_idx, f"region_{region_idx}"),
 "calibration_score": float(calibration_arr[expert_idx, region_idx]),
 })
else:
 calibration_rows = []
 for idx, value in enumerate(calibration_arr.reshape(-1)):
 calibration_rows.append({
 "expert_id": None,
 "expert_name": None,
 "region_id": None,
 "region_name": None,
 "calibration_index": idx,
 "calibration_score": float(value),
 })

calibration_df = pd.DataFrame(calibration_rows)
calibration_path = os.path.join(EXPERT_ANALYSIS_DIR, "router_calibration_scores_by_region.csv")
calibration_df.to_csv(calibration_path, index=False)

if log_tau_tensor is not None:
 log_tau_value = float(log_tau_tensor.reshape(-1)[0].item())
 tau_value = float(math.exp(log_tau_value))
else:
 log_tau_value = float("nan")
 tau_value = float("nan")

static_router_values = {
 "checkpoint_path": EXPERT_CHECKPOINT_PATH,
 "log_tau": log_tau_value,
 "tau": tau_value,
 "lambda_calib": float(getattr(cfg_v2_eval, "router_calibration_lambda", float("nan"))),
 "calibration_shape": list(calibration_tensor.shape),
 "score_mlp_last_weight_shape": list(score_layer_w.shape) if score_layer_w is not None else None,
 "score_mlp_last_bias": score_layer_b.reshape(-1).tolist() if score_layer_b is not None else None,
 "score_mlp_first_weight_shape": list(first_layer_w.shape) if first_layer_w is not None else None,
 "region_embedding_shape": list(region_embed.shape) if region_embed is not None else None,
}

static_values_path = os.path.join(EXPERT_ANALYSIS_DIR, "static_router_values.json")
with open(static_values_path, "w") as f:
 json.dump(static_router_values, f, indent=2)

if score_layer_w is not None:
 score_layer_w_df = pd.DataFrame(
 score_layer_w.numpy(),
 index=[f"score_output_{name}" for name in EXPERT_NAMES[:score_layer_w.shape[0]]],
 )
 score_layer_w_path = os.path.join(EXPERT_ANALYSIS_DIR, "router_score_mlp_last_layer_weight.csv")
 score_layer_w_df.to_csv(score_layer_w_path)
else:
 score_layer_w_path = None

if score_layer_b is not None:
 score_layer_b_df = pd.DataFrame({
 "expert_id": np.arange(score_layer_b.numel()) + 1,
 "expert_name": EXPERT_NAMES[:score_layer_b.numel()],
 "score_mlp_last_layer_bias": score_layer_b.reshape(-1).numpy(),
 })
 score_layer_b_path = os.path.join(EXPERT_ANALYSIS_DIR, "router_score_mlp_last_layer_bias.csv")
 score_layer_b_df.to_csv(score_layer_b_path, index=False)
else:
 score_layer_b_path = None

print("=" * 80)
print("Static router values")
print("=" * 80)
print(json.dumps(static_router_values, indent=2))
print(f"Saved calibration table: {calibration_path}")
print(f"Saved static values JSON: {static_values_path}")
if score_layer_w_path:
 print(f"Saved score MLP final layer weights: {score_layer_w_path}")
if score_layer_b_path:
 print(f"Saved score MLP final layer bias: {score_layer_b_path}")

display(calibration_df)

In [ ]:
# Build V2 eval loaders if they are not already available

if "v2_test_loader" not in globals() or "v2_val_loader" not in globals():
 print("=" * 80)
 print("Building V2 evaluation loaders for expert-weight analysis")
 print("=" * 80)

 if "build_v2_eval_loaders_matching_v1" in globals():
 needed_names = ["cfg_v2_eval", "val_ids_v2", "test_ids_v2"]
 missing_names = [name for name in needed_names if name not in globals()]

 if missing_names:
 raise NameError(
 "Cannot build V2 evaluation loaders because these variables are missing: "
 + ", ".join(missing_names)
 + "\nRun the V2 evaluation setup cells before this cell."
 )

 v2_val_loader, v2_test_loader, v2_missing_val_df, v2_missing_test_df = (
 build_v2_eval_loaders_matching_v1(cfg_v2_eval, val_ids_v2, test_ids_v2)
 )

 elif "build_dataloaders_v2" in globals():
 needed_names = ["cfg_v2_eval"]
 missing_names = [name for name in needed_names if name not in globals()]

 if missing_names:
 raise NameError(
 "Cannot build V2 dataloaders because these variables are missing: "
 + ", ".join(missing_names)
 + "\nRun the V2 config/setup cells before this cell."
 )

 _, v2_val_loader, v2_test_loader, v2_splits = build_dataloaders_v2(cfg_v2_eval)

 else:
 raise NameError(
 "No V2 loader-building function was found. "
 "Run the cells that define build_v2_eval_loaders_matching_v1 or build_dataloaders_v2."
 )

print("=" * 80)
print("Available expert-weight loaders")
print("=" * 80)
print(f"v2_val_loader: {'yes' if 'v2_val_loader' in globals() else 'no'}")
print(f"v2_test_loader: {'yes' if 'v2_test_loader' in globals() and v2_test_loader is not None else 'no'}")

In [ ]:
# Inference-time expert weight collection by case and anatomy region

@torch.no_grad()
def collect_v2_expert_weights(model, loader, cfg, split_name="test", max_batches=None):
 model.eval()

 case_rows = []
 region_rows = []

 for batch_idx, batch in enumerate(tqdm(loader, desc=f"Expert weights {split_name}", leave=True)):
 if max_batches is not None and batch_idx >= max_batches:
 break

 image = batch["image"].to(cfg.device, non_blocking=True)
 priors = batch["priors"].to(cfg.device, non_blocking=True)
 region = batch["region"].to(cfg.device, non_blocking=True)

 outputs = model(image, priors, region)

 image_weights = outputs["weights"].detach().float().cpu().numpy()
 region_weights = outputs["region_weights"].detach().float().cpu().numpy()
 router_scores = outputs["router_scores"].detach().float().cpu().numpy()

 prepared_region = model._prepare_region(region, image.shape[-3:]).detach().cpu().numpy()
 batch_size = image_weights.shape[0]

 for b in range(batch_size):
 if "v2_extract_case_id" in globals():
 case_id = v2_extract_case_id(batch)
 if isinstance(case_id, (list, tuple)):
 case_id = case_id[b]
 else:
 raw_case_id = batch.get("case_id", f"{split_name}_{batch_idx:04d}")
 case_id = raw_case_id[b] if isinstance(raw_case_id, (list, tuple)) else raw_case_id

 case_row = {
 "split": split_name,
 "case_id": str(case_id),
 "batch_index": int(batch_idx),
 }

 for expert_idx, expert_name in enumerate(EXPERT_NAMES):
 case_row[f"image_weight_{expert_name}"] = float(image_weights[b, expert_idx])

 case_rows.append(case_row)

 region_ids, region_counts = np.unique(prepared_region[b].reshape(-1), return_counts=True)
 total_voxels = float(np.sum(region_counts))

 region_fraction_map = {
 int(region_id): float(count / max(total_voxels, 1.0))
 for region_id, count in zip(region_ids, region_counts)
 }

 for region_id in range(region_weights.shape[1]):
 region_row = {
 "split": split_name,
 "case_id": str(case_id),
 "batch_index": int(batch_idx),
 "region_id": int(region_id),
 "region_name": REGION_NAMES.get(region_id, f"region_{region_id}"),
 "region_fraction": region_fraction_map.get(region_id, 0.0),
 }

 for expert_idx, expert_name in enumerate(EXPERT_NAMES):
 region_row[f"region_weight_{expert_name}"] = float(region_weights[b, region_id, expert_idx])
 region_row[f"router_score_{expert_name}"] = float(router_scores[b, region_id, expert_idx])

 region_rows.append(region_row)

 del outputs, image, priors, region

 if torch.cuda.is_available():
 torch.cuda.empty_cache()

 case_df = pd.DataFrame(case_rows)
 region_df = pd.DataFrame(region_rows)

 return case_df, region_df


# Full test split by default.
# Change EXPERT_MAX_BATCHES to a small integer like 5 for a quick check.
EXPERT_SPLIT_NAME = "test"
EXPERT_MAX_BATCHES = None

EXPERT_LOADER = v2_test_loader
EXPERT_MODEL = v2_best_model
EXPERT_CFG = cfg_v2_eval

EXPERT_MODEL.eval()

expert_case_weights_df, expert_region_weights_df = collect_v2_expert_weights(
 model=EXPERT_MODEL,
 loader=EXPERT_LOADER,
 cfg=EXPERT_CFG,
 split_name=EXPERT_SPLIT_NAME,
 max_batches=EXPERT_MAX_BATCHES,
)

case_weights_path = os.path.join(EXPERT_ANALYSIS_DIR, f"{EXPERT_SPLIT_NAME}_case_expert_weights.csv")
region_weights_path = os.path.join(EXPERT_ANALYSIS_DIR, f"{EXPERT_SPLIT_NAME}_region_expert_weights.csv")

expert_case_weights_df.to_csv(case_weights_path, index=False)
expert_region_weights_df.to_csv(region_weights_path, index=False)

print("=" * 80)
print("Inference-time expert weights")
print("=" * 80)
print("Model used: v2_best_model")
print("Loader used: v2_test_loader")
print("Config used: cfg_v2_eval")
print(f"Cases analyzed: {len(expert_case_weights_df)}")
print(f"Saved case-level weights: {case_weights_path}")
print(f"Saved region-level weights: {region_weights_path}")

display(expert_case_weights_df.head())
display(expert_region_weights_df.head())

In [ ]:
# Inspect availability of TM, IM, IT, and sequence-length related values

import os
import json
import torch
import pandas as pd
import numpy as np

SEARCH_TERMS = [
 "tm", "im", "it",
 "sequence", "seq", "length", "len",
 "time", "timesteps", "tokens", "patches",
 "memory", "iteration", "image"
]

def value_summary(value):
 if torch.is_tensor(value):
 return {
 "type": "torch.Tensor",
 "shape": list(value.shape),
 "dtype": str(value.dtype),
 }
 if isinstance(value, np.ndarray):
 return {
 "type": "np.ndarray",
 "shape": list(value.shape),
 "dtype": str(value.dtype),
 }
 if isinstance(value, (list, tuple)):
 return {
 "type": type(value).__name__,
 "length": len(value),
 }
 if isinstance(value, dict):
 return {
 "type": "dict",
 "num_keys": len(value),
 }
 return {
 "type": type(value).__name__,
 "value": str(value)[:120],
 }

def matches_terms(name, terms=SEARCH_TERMS):
 lowered = str(name).lower()
 return any(term in lowered for term in terms)

def inspect_object_attrs(obj, object_name):
 rows = []
 if obj is None:
 return rows

 for name in dir(obj):
 if name.startswith("__"):
 continue
 if not matches_terms(name):
 continue

 try:
 value = getattr(obj, name)
 summary = value_summary(value)
 except Exception as exc:
 summary = {"type": "unreadable", "value": str(exc)}

 rows.append({
 "source": object_name,
 "name": name,
 **summary,
 })

 return rows

def inspect_dict_keys(d, object_name):
 rows = []
 if not isinstance(d, dict):
 return rows

 for name, value in d.items():
 if not matches_terms(name):
 continue

 rows.append({
 "source": object_name,
 "name": name,
 **value_summary(value),
 })

 return rows

inspection_rows = []

# Config values
if "cfg_v2_eval" in globals():
 inspection_rows.extend(inspect_object_attrs(cfg_v2_eval, "cfg_v2_eval"))

# Model attributes
if "v2_best_model" in globals():
 inspection_rows.extend(inspect_object_attrs(v2_best_model, "v2_best_model"))

 for module_name, module in v2_best_model.named_modules():
 if matches_terms(module_name):
 inspection_rows.append({
 "source": "v2_best_model.named_modules",
 "name": module_name,
 "type": type(module).__name__,
 "value": "",
 })

# Checkpoint keys
if "EXPERT_CHECKPOINT_PATH" in globals() and os.path.exists(EXPERT_CHECKPOINT_PATH):
 try:
 ckpt = torch.load(EXPERT_CHECKPOINT_PATH, map_location="cpu", weights_only=False)
 except TypeError:
 ckpt = torch.load(EXPERT_CHECKPOINT_PATH, map_location="cpu")

 inspection_rows.extend(inspect_dict_keys(ckpt, "checkpoint"))

 if isinstance(ckpt, dict):
 state = ckpt.get("model_state", ckpt.get("state_dict", None))
 if isinstance(state, dict):
 for key, value in state.items():
 if matches_terms(key):
 inspection_rows.append({
 "source": "checkpoint.model_state",
 "name": key,
 **value_summary(value),
 })

# Batch keys
if "v2_test_loader" in globals():
 batch = next(iter(v2_test_loader))
 inspection_rows.extend(inspect_dict_keys(batch, "v2_test_loader.batch"))

 for key, value in batch.items():
 if torch.is_tensor(value):
 inspection_rows.append({
 "source": "v2_test_loader.batch_shape",
 "name": key,
 **value_summary(value),
 })

inspection_df = pd.DataFrame(inspection_rows)

print("=" * 80)
print("TM, IM, IT, and sequence-length availability inspection")
print("=" * 80)

if len(inspection_df) == 0:
 print("No direct matches found from config, checkpoint, model attributes, or batch keys.")
else:
 display(inspection_df)

In [ ]:
# Inspect one forward pass for output keys and tensor shapes

@torch.no_grad()
def inspect_one_v2_forward_pass(model, loader, cfg):
 model.eval()

 batch = next(iter(loader))

 image = batch["image"].to(cfg.device, non_blocking=True)
 priors = batch["priors"].to(cfg.device, non_blocking=True)
 region = batch["region"].to(cfg.device, non_blocking=True)

 outputs = model(image, priors, region)

 rows = []

 for key, value in batch.items():
 rows.append({
 "source": "batch",
 "name": key,
 **value_summary(value),
 })

 for key, value in outputs.items():
 rows.append({
 "source": "model_outputs",
 "name": key,
 **value_summary(value),
 })

 output_df = pd.DataFrame(rows)

 print("=" * 80)
 print("One-pass batch and model-output inspection")
 print("=" * 80)
 display(output_df)

 return batch, outputs, output_df

one_batch, one_outputs, one_pass_df = inspect_one_v2_forward_pass(
 model=v2_best_model,
 loader=v2_test_loader,
 cfg=cfg_v2_eval,
)

In [ ]:
# Infer sequence length from tensors likely to contain token, patch, or region sequences

def infer_sequence_length_from_tensor(name, tensor):
 if not torch.is_tensor(tensor):
 return None

 shape = list(tensor.shape)

 # Common layouts:
 # [B, N, C] means N is sequence length
 # [B, C, D, H, W] means D*H*W can be treated as voxel sequence length
 # [B, C, H, W] means H*W can be treated as spatial sequence length
 if len(shape) == 3:
 return {
 "name": name,
 "shape": shape,
 "inferred_sequence_length": int(shape[1]),
 "interpretation": "B x N x C",
 }

 if len(shape) == 5:
 return {
 "name": name,
 "shape": shape,
 "inferred_sequence_length": int(shape[2] * shape[3] * shape[4]),
 "interpretation": "B x C x D x H x W flattened spatial volume",
 }

 if len(shape) == 4:
 return {
 "name": name,
 "shape": shape,
 "inferred_sequence_length": int(shape[2] * shape[3]),
 "interpretation": "B x C x H x W flattened spatial map",
 }

 return {
 "name": name,
 "shape": shape,
 "inferred_sequence_length": None,
 "interpretation": "No standard sequence interpretation",
 }

seq_rows = []

for key, value in one_batch.items():
 if torch.is_tensor(value):
 inferred = infer_sequence_length_from_tensor(f"batch.{key}", value)
 if inferred is not None:
 seq_rows.append(inferred)

for key, value in one_outputs.items():
 if torch.is_tensor(value):
 inferred = infer_sequence_length_from_tensor(f"outputs.{key}", value)
 if inferred is not None:
 seq_rows.append(inferred)

sequence_length_df = pd.DataFrame(seq_rows)

print("=" * 80)
print("Inferred sequence-length candidates")
print("=" * 80)
display(sequence_length_df)

In [ ]:
# Aggregate expert-weight CSVs for paper tables

import os
import json
import pandas as pd

EXPERT_ANALYSIS_DIR = "FILL IN WITH DIRECTORY FOR EVALUATION RESULTS_v2/expert_weight_analysis"

case_csv_path = os.path.join(EXPERT_ANALYSIS_DIR, "test_case_expert_weights.csv")
region_csv_path = os.path.join(EXPERT_ANALYSIS_DIR, "test_region_expert_weights.csv")

case_df = pd.read_csv(case_csv_path)
region_df = pd.read_csv(region_csv_path)

# Global image-level expert weights
global_summary = case_df[
 [
 "image_weight_context_expert",
 "image_weight_boundary_expert",
 "image_weight_anatomy_expert",
 ]
].agg(["mean", "std", "min", "max"])

# Cleaner paper table version
global_summary_table = (
 global_summary.T
 .reset_index()
 .rename(columns={
 "index": "expert",
 "mean": "mean_weight",
 "std": "std_weight",
 "min": "min_weight",
 "max": "max_weight",
 })
)

global_summary_table["expert"] = (
 global_summary_table["expert"]
 .str.replace("image_weight_", "", regex=False)
 .str.replace("_expert", "", regex=False)
)

# Region-level expert weights
region_summary = (
 region_df
 .groupby("region_name")
 .agg(
 region_fraction=("region_fraction", "mean"),
 context_weight=("region_weight_context_expert", "mean"),
 boundary_weight=("region_weight_boundary_expert", "mean"),
 anatomy_weight=("region_weight_anatomy_expert", "mean"),
 context_std=("region_weight_context_expert", "std"),
 boundary_std=("region_weight_boundary_expert", "std"),
 anatomy_std=("region_weight_anatomy_expert", "std"),
 )
 .reset_index()
)

# Optional region ordering
region_order = [
 "background_or_other",
 "left_white_matter",
 "right_white_matter",
 "cortex",
 "periventricular",
 "deep_gray",
]

region_summary["region_name"] = pd.Categorical(
 region_summary["region_name"],
 categories=region_order,
 ordered=True,
)
region_summary = region_summary.sort_values("region_name").reset_index(drop=True)
region_summary["region_name"] = region_summary["region_name"].astype(str)

# Save aggregate outputs
global_summary_path = os.path.join(EXPERT_ANALYSIS_DIR, "test_global_expert_weight_summary_for_tables.csv")
region_summary_path = os.path.join(EXPERT_ANALYSIS_DIR, "test_region_expert_weight_summary_for_tables.csv")

global_summary_table.to_csv(global_summary_path, index=False)
region_summary.to_csv(region_summary_path, index=False)

paper_summary = {
 "num_cases": int(len(case_df)),
 "num_region_rows": int(len(region_df)),
 "global_summary_csv": global_summary_path,
 "region_summary_csv": region_summary_path,
 "dominant_global_expert": global_summary_table.sort_values("mean_weight", ascending=False).iloc[0]["expert"],
 "dominant_global_mean_weight": float(global_summary_table["mean_weight"].max()),
}

paper_summary_path = os.path.join(EXPERT_ANALYSIS_DIR, "test_expert_weight_table_summary.json")
with open(paper_summary_path, "w") as f:
 json.dump(paper_summary, f, indent=2)

print("=" * 80)
print("Global image-level expert weights")
print("=" * 80)
display(global_summary_table)

print("\nMarkdown table: global image-level expert weights")
print(global_summary_table.to_markdown(index=False, floatfmt=".6f"))

print("\n" + "=" * 80)
print("Region-level expert weights")
print("=" * 80)
display(region_summary)

print("\nMarkdown table: region-level expert weights")
print(region_summary.to_markdown(index=False, floatfmt=".6f"))

print("\n" + "=" * 80)
print("Saved aggregate files")
print("=" * 80)
print(f"Global summary CSV: {global_summary_path}")
print(f"Region summary CSV: {region_summary_path}")
print(f"JSON summary: {paper_summary_path}")